In [50]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_root_files = (candidate / "train.csv").exists() and (candidate / "test.csv").exists()
        has_data_files = (candidate / "data" / "train.csv").exists() and (candidate / "data" / "test.csv").exists()
        if has_root_files or has_data_files:
            return candidate
    raise FileNotFoundError("Could not locate project root.")


PROJECT_ROOT = find_project_root()


def resolve_data_path(filename: str) -> Path:
    for path in [PROJECT_ROOT / filename, PROJECT_ROOT / "data" / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename} in project root or data/")


train = pd.read_csv(resolve_data_path("train.csv"), parse_dates=["date"])
test = pd.read_csv(resolve_data_path("test.csv"), parse_dates=["date"])

for df in (train, test):
    if "tau" not in df.columns:
        df["tau"] = df["maturity_days"] / 365.0

train_date_summary = (
    train.groupby("date")
    .agg(
        total_rows=("row_id", "size"),
        observed_rows=("iv_observed", lambda s: s.notna().sum()),
        missing_rows=("iv_observed", lambda s: s.isna().sum()),
        observed_ratio=("iv_observed", lambda s: s.notna().mean()),
    )
    .sort_index()
)

train_dates = train_date_summary.index.to_list()

N_FOLDS = 4
VAL_DATES_PER_FOLD = 5
TOTAL_VAL_DATES = N_FOLDS * VAL_DATES_PER_FOLD
val_start_idx = len(train_dates) - TOTAL_VAL_DATES

fold_plan = pd.DataFrame(
    [
        {
            "fold": fold_id + 1,
            "train_start": train_dates[0],
            "train_end": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD - 1],
            "n_train_dates": val_start_idx + fold_id * VAL_DATES_PER_FOLD,
            "val_start": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD],
            "val_end": train_dates[val_start_idx + (fold_id + 1) * VAL_DATES_PER_FOLD - 1],
            "n_val_dates": VAL_DATES_PER_FOLD,
        }
        for fold_id in range(N_FOLDS)
    ]
)

print("Project root:", PROJECT_ROOT)
print("Train shape:", train.shape, "| Test shape:", test.shape)
display(fold_plan)


Project root: /Users/sauravkrishna/Documents/projects/NQFO-Impilied-volatility-surface
Train shape: (11640, 10) | Test shape: (3960, 10)


,fold,train_start,train_end,n_train_dates,val_start,val_end,n_val_dates
0,1,2025-01-02,2025-04-18,77,2025-04-21,2025-04-25,5
1,2,2025-01-02,2025-04-25,82,2025-04-28,2025-05-02,5
2,3,2025-01-02,2025-05-02,87,2025-05-05,2025-05-09,5
3,4,2025-01-02,2025-05-09,92,2025-05-12,2025-05-16,5


In [51]:
BUCKET_COLS = ["maturity_label", "option_type"]
NODE_COLS = ["maturity_label", "moneyness", "option_type"]

MASK_PROTOCOL_CONFIG = {
    "stress_test": {
        "base_hide_rate": 0.10,
        "node_weight": 1.00,
        "support_weight": 0.00,
    },
    "primary_realistic": {
        "base_hide_rate": 0.10,
        "node_weight": 0.65,
        "support_weight": 0.35,
    },
}

LOCKED_MASK_SEED = "nqfo-val-v1"
PHASE4_SHRINK_ALPHA = 0.25

OVERALL_TEST_MISSING_RATE = test["iv_observed"].isna().mean()

test_bucket_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(BUCKET_COLS)["is_missing"]
    .mean()
    .rename("test_bucket_missing_rate")
    .reset_index()
)

test_node_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(NODE_COLS)["is_missing"]
    .mean()
    .rename("test_node_missing_rate")
    .reset_index()
)

surface_levels = pd.concat(
    [
        train[["moneyness", "maturity_label", "maturity_days"]],
        test[["moneyness", "maturity_label", "maturity_days"]],
    ],
    ignore_index=True,
)

moneyness_levels = sorted(surface_levels["moneyness"].dropna().unique().tolist())
maturity_levels = (
    surface_levels[["maturity_label", "maturity_days"]]
    .drop_duplicates()
    .sort_values("maturity_days")["maturity_label"]
    .tolist()
)

m_idx = {m: i for i, m in enumerate(moneyness_levels)}
t_idx = {t: i for i, t in enumerate(maturity_levels)}


def stable_uniform(key: str) -> float:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def opposite_option(opt: str) -> str:
    return "put" if opt == "call" else "call"


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))


def score_predictions(scored_df: pd.DataFrame, pred_col: str = "iv_pred") -> dict:
    eval_df = scored_df.loc[scored_df["is_scored_target"]].copy()
    eval_df = eval_df.loc[eval_df["iv_observed"].notna() & eval_df[pred_col].notna()].copy()
    return {
        "n_scored": len(eval_df),
        "rmse": rmse(eval_df["iv_observed"], eval_df[pred_col]) if len(eval_df) else np.nan,
    }


def local_support_profile(target_rows: pd.DataFrame, visible_rows: pd.DataFrame) -> pd.DataFrame:
    prof = target_rows.copy()

    visible_key_set = set(
        zip(
            visible_rows["date"],
            visible_rows["moneyness"],
            visible_rows["maturity_label"],
            visible_rows["option_type"],
        )
    )

    opp_visible = []
    same_maturity_adj_count = []
    same_moneyness_adj_count = []

    for d, m, t, o in zip(
        prof["date"],
        prof["moneyness"],
        prof["maturity_label"],
        prof["option_type"],
    ):
        opp_visible.append((d, m, t, opposite_option(o)) in visible_key_set)

        i = m_idx[m]
        j = t_idx[t]

        same_maturity_candidates = []
        if i - 1 >= 0:
            same_maturity_candidates.append((d, moneyness_levels[i - 1], t, o))
        if i + 1 < len(moneyness_levels):
            same_maturity_candidates.append((d, moneyness_levels[i + 1], t, o))

        same_moneyness_candidates = []
        if j - 1 >= 0:
            same_moneyness_candidates.append((d, m, maturity_levels[j - 1], o))
        if j + 1 < len(maturity_levels):
            same_moneyness_candidates.append((d, m, maturity_levels[j + 1], o))

        same_maturity_adj_count.append(sum(c in visible_key_set for c in same_maturity_candidates))
        same_moneyness_adj_count.append(sum(c in visible_key_set for c in same_moneyness_candidates))

    prof["opp_option_visible"] = opp_visible
    prof["same_maturity_adj_visible_count"] = same_maturity_adj_count
    prof["same_moneyness_adj_visible_count"] = same_moneyness_adj_count
    prof["local_support_score"] = (
        prof["opp_option_visible"].astype(int)
        + prof["same_maturity_adj_visible_count"]
        + prof["same_moneyness_adj_visible_count"]
    )
    return prof


def build_masked_validation_rows_with_protocol(
    full_df: pd.DataFrame,
    target_dates,
    protocol_name: str,
    seed: str = LOCKED_MASK_SEED,
) -> pd.DataFrame:
    cfg = MASK_PROTOCOL_CONFIG[protocol_name]

    val_df = full_df.loc[full_df["date"].isin(target_dates)].copy()
    val_df["is_orig_observed"] = val_df["iv_observed"].notna()
    val_df["is_orig_missing"] = ~val_df["is_orig_observed"]

    val_df = val_df.merge(test_bucket_pattern, on=BUCKET_COLS, how="left")
    val_df = val_df.merge(test_node_pattern, on=NODE_COLS, how="left")

    val_df["bucket_hide_rate_on_observed"] = (
        cfg["base_hide_rate"] * val_df["test_bucket_missing_rate"] / OVERALL_TEST_MISSING_RATE
    )

    val_df["priority_noise"] = val_df.apply(
        lambda r: stable_uniform(
            f"{seed}|{protocol_name}|{r['date'].date()}|{r['maturity_label']}|{r['moneyness']:.4f}|{r['option_type']}"
        ),
        axis=1,
    )

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    observed_support = local_support_profile(observed_pool, observed_pool)[["row_id", "local_support_score"]]
    val_df = val_df.merge(observed_support, on="row_id", how="left")
    val_df["local_support_score"] = val_df["local_support_score"].fillna(0).astype(int)

    val_df["is_pseudo_hidden"] = False

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    for _, g in observed_pool.groupby(["date", *BUCKET_COLS], sort=False):
        n_obs = len(g)
        n_hide = int(np.round(g["bucket_hide_rate_on_observed"].iloc[0] * n_obs))
        if n_hide <= 0:
            continue

        node_rank = g["test_node_missing_rate"].rank(method="average", pct=True)
        support_rank = g["local_support_score"].rank(method="average", pct=True)

        selection_priority = (
            cfg["node_weight"] * node_rank
            + cfg["support_weight"] * support_rank
            + 1e-6 * g["priority_noise"]
        )

        chosen_idx = (
            g.assign(selection_priority=selection_priority)
            .sort_values(["selection_priority", "row_id"], ascending=[False, True])
            .head(n_hide)
            .index
        )
        val_df.loc[chosen_idx, "is_pseudo_hidden"] = True

    val_df["is_scored_target"] = val_df["is_pseudo_hidden"]
    val_df["is_visible_anchor"] = val_df["is_orig_observed"] & ~val_df["is_pseudo_hidden"]
    val_df["is_effectively_missing"] = val_df["is_orig_missing"] | val_df["is_pseudo_hidden"]
    val_df["mask_protocol"] = protocol_name

    return val_df.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)


In [52]:
def build_node_lookup(observed_df: pd.DataFrame, pred_name: str) -> pd.DataFrame:
    return (
        observed_df.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename(pred_name)
        .reset_index()
    )


def predict_recent_node_median(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = target_rows.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    recent_dates = sorted(observed_train["date"].unique())[-lookback_dates:]
    recent_obs = observed_train.loc[observed_train["date"].isin(recent_dates)].copy()

    recent_lookup = build_node_lookup(recent_obs, "recent_node_pred")
    full_lookup = build_node_lookup(observed_train, "full_node_pred")

    out = out.merge(recent_lookup, on=NODE_COLS, how="left")
    out = out.merge(full_lookup, on=NODE_COLS, how="left")

    out["iv_pred"] = out["recent_node_pred"].fillna(out["full_node_pred"]).fillna(global_median)
    out["pred_source"] = np.select(
        [
            out["recent_node_pred"].notna(),
            out["full_node_pred"].notna(),
        ],
        [
            "recent_node_median",
            "full_node_median",
        ],
        default="global_median",
    )
    return out


def predict_same_date_linear_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, target_rows, lookback_dates=lookback_dates).copy()

    out["interp_pred"] = np.nan
    out["interp_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        x = anchors["moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        if len(anchors) == 1:
            interp_vals = np.repeat(y[0], len(g))
            interp_label = "same_date_single_anchor"
        else:
            interp_vals = np.interp(g["moneyness"].to_numpy(), x, y)
            interp_label = "same_date_linear_interp"

        out.loc[g.index, "interp_pred"] = interp_vals
        out.loc[g.index, "interp_source"] = interp_label

    use_interp = out["interp_pred"].notna()
    out.loc[use_interp, "iv_pred"] = out.loc[use_interp, "interp_pred"]
    out["pred_source"] = np.where(use_interp, out["interp_source"], out["pred_source"])

    return out


def predict_quadratic_smile_logm(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    degree: int = 2,
    min_anchors: int = 5,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["log_moneyness"] = np.log(out["moneyness"])
    out["smile_pred"] = np.nan
    out["smile_anchor_count"] = 0
    out["pred_source_smile"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["log_moneyness", "iv_observed"]]
            .dropna()
            .sort_values("log_moneyness")
        )

        if len(anchors) < min_anchors:
            continue
        if anchors["log_moneyness"].nunique() < degree + 1:
            continue

        x = anchors["log_moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        x_center = x.mean()
        coeffs = np.polyfit(x - x_center, y, deg=degree)

        target_x = g["log_moneyness"].to_numpy()
        pred = np.polyval(coeffs, target_x - x_center)

        in_range = (target_x >= x.min()) & (target_x <= x.max())
        pred = np.where(in_range, pred, np.nan)
        pred = np.clip(pred, pred_lo, pred_hi)

        out.loc[g.index, "smile_pred"] = pred
        out.loc[g.index, "smile_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_smile"] = np.where(in_range, "quadratic_smile_logm", pd.NA)

    use_smile = out["smile_pred"].notna()
    out.loc[use_smile, "iv_pred"] = out.loc[use_smile, "smile_pred"]
    out["pred_source"] = np.where(use_smile, out["pred_source_smile"], out["pred_source"])

    return out


def predict_total_variance_maturity_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["anchor_total_variance"] = np.where(
        out["iv_observed"].notna(),
        (out["iv_observed"] / 100.0) ** 2 * out["tau"],
        np.nan,
    )
    out["tv_pred"] = np.nan
    out["tv_anchor_count"] = 0
    out["pred_source_tv"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, moneyness, option_type), g_idx in out.groupby(
        ["date", "moneyness", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["tau", "anchor_total_variance"]]
            .dropna()
            .sort_values("tau")
        )

        if len(anchors) < 2:
            continue

        x = anchors["tau"].to_numpy()
        y = anchors["anchor_total_variance"].to_numpy()
        target_tau = g["tau"].to_numpy()

        in_range = (target_tau >= x.min()) & (target_tau <= x.max())
        pred_tv = np.interp(target_tau, x, y)
        pred_tv = np.where(in_range, pred_tv, np.nan)
        pred_tv = np.where(pred_tv > 0, pred_tv, np.nan)

        pred_iv = 100.0 * np.sqrt(pred_tv / target_tau)
        pred_iv = np.clip(pred_iv, pred_lo, pred_hi)

        out.loc[g.index, "tv_pred"] = pred_iv
        out.loc[g.index, "tv_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_tv"] = np.where(in_range, "tv_maturity_interp", pd.NA)

    use_tv = out["tv_pred"].notna()
    out.loc[use_tv, "iv_pred"] = out.loc[use_tv, "tv_pred"]
    out["pred_source"] = np.where(use_tv, out["pred_source_tv"], out["pred_source"])

    return out


def predict_structured_region_blend(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    smile_weight_center: float = 0.65,
    smile_weight_wing: float = 0.35,
) -> pd.DataFrame:
    base = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    smile = predict_quadratic_smile_logm(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "smile_pred"]].copy()

    tv = predict_total_variance_maturity_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "tv_pred"]].copy()

    out = base.merge(smile, on="row_id", how="left")
    out = out.merge(tv, on="row_id", how="left")

    out["wing_center_bucket"] = np.where(
        out["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )

    both_available = out["smile_pred"].notna() & out["tv_pred"].notna()
    only_smile = out["smile_pred"].notna() & out["tv_pred"].isna()
    only_tv = out["tv_pred"].notna() & out["smile_pred"].isna()

    center_mask = both_available & (out["wing_center_bucket"] == "center")
    wing_mask = both_available & (out["wing_center_bucket"] == "wing")

    out.loc[center_mask, "iv_pred"] = (
        smile_weight_center * out.loc[center_mask, "smile_pred"]
        + (1.0 - smile_weight_center) * out.loc[center_mask, "tv_pred"]
    )
    out.loc[wing_mask, "iv_pred"] = (
        smile_weight_wing * out.loc[wing_mask, "smile_pred"]
        + (1.0 - smile_weight_wing) * out.loc[wing_mask, "tv_pred"]
    )
    out.loc[only_smile, "iv_pred"] = out.loc[only_smile, "smile_pred"]
    out.loc[only_tv, "iv_pred"] = out.loc[only_tv, "tv_pred"]

    out["pred_source"] = np.select(
        [
            center_mask,
            wing_mask,
            only_smile,
            only_tv,
        ],
        [
            "structured_region_blend_center",
            "structured_region_blend_wing",
            "quadratic_smile_only",
            "tv_maturity_only",
        ],
        default=out["pred_source"],
    )

    return out


def predict_structured_region_blend_callput_shrink(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    shrink_alpha: float = PHASE4_SHRINK_ALPHA,
) -> pd.DataFrame:
    out = predict_structured_region_blend(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    opposite_visible = out.loc[
        out["is_visible_anchor"],
        ["date", "moneyness", "maturity_label", "option_type", "iv_observed"],
    ].copy()
    opposite_visible["option_type"] = opposite_visible["option_type"].map(opposite_option)
    opposite_visible = opposite_visible.rename(columns={"iv_observed": "opp_visible_iv"})

    out = out.merge(
        opposite_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    use_shrink = out["opp_visible_iv"].notna()
    out.loc[use_shrink, "iv_pred"] = (
        (1.0 - shrink_alpha) * out.loc[use_shrink, "iv_pred"]
        + shrink_alpha * out.loc[use_shrink, "opp_visible_iv"]
    )

    out["pred_source"] = np.where(
        use_shrink,
        "structured_region_blend_callput_shrink",
        out["pred_source"],
    )

    return out


print("Phase 5.0 setup ready.")


Phase 5.0 setup ready.


In [53]:
ML_MIN_HISTORY_DATES = 20
ML_MASK_DATES_PER_FOLD = 20


def get_ml_candidate_dates_for_fold(
    fold_row,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
):
    fold_train_dates = train_date_summary.loc[:fold_row.train_end].index.to_list()
    eligible = fold_train_dates[min_history_dates:]
    return eligible[-n_mask_dates:]


def make_single_date_training_bundle(
    train_df: pd.DataFrame,
    target_date,
    protocol_name: str,
) -> dict:
    history_df = train_df.loc[train_df["date"] < target_date].copy()
    masked_target_date = build_masked_validation_rows_with_protocol(
        train_df,
        [target_date],
        protocol_name=protocol_name,
    )

    if history_df.empty:
        raise ValueError("History dataframe is empty. Choose a later target date.")

    if not (history_df["date"].max() < pd.Timestamp(target_date)):
        raise ValueError("Temporal leakage detected: history includes target date or later.")

    return {
        "target_date": pd.Timestamp(target_date),
        "protocol": protocol_name,
        "train_history": history_df,
        "masked_target_date": masked_target_date,
    }


fold1 = fold_plan.iloc[0]
fold1_ml_dates = get_ml_candidate_dates_for_fold(fold1)

print("Fold 1 ML candidate dates")
display(pd.Series(fold1_ml_dates, name="target_date").to_frame())

demo_date = fold1_ml_dates[-1]
demo_bundle = make_single_date_training_bundle(
    train,
    target_date=demo_date,
    protocol_name="primary_realistic",
)

demo_pred = predict_structured_region_blend_callput_shrink(
    demo_bundle["train_history"],
    demo_bundle["masked_target_date"],
    lookback_dates=20,
    shrink_alpha=PHASE4_SHRINK_ALPHA,
)

demo_metrics = score_predictions(demo_pred)

demo_summary = pd.Series(
    {
        "target_date": demo_bundle["target_date"].date().isoformat(),
        "protocol": demo_bundle["protocol"],
        "n_history_dates": demo_bundle["train_history"]["date"].nunique(),
        "history_end_date": demo_bundle["train_history"]["date"].max().date().isoformat(),
        "n_rows_target_date": len(demo_pred),
        "n_visible_anchors": int(demo_pred["is_visible_anchor"].sum()),
        "n_scored_targets": int(demo_pred["is_scored_target"].sum()),
        "n_orig_missing": int(demo_pred["is_orig_missing"].sum()),
        "demo_rmse": demo_metrics["rmse"],
        "all_scored_have_preds": bool(demo_pred.loc[demo_pred["is_scored_target"], "iv_pred"].notna().all()),
    }
)

print("Single-date leakage-safe training-bundle check")
display(demo_summary.to_frame("value"))

print("Prediction source mix on scored targets")
display(
    demo_pred.loc[demo_pred["is_scored_target"], "pred_source"]
    .value_counts(dropna=False)
    .rename_axis("pred_source")
    .to_frame("count")
)
print("Scored target preview")
display(
    demo_pred.loc[demo_pred["is_scored_target"]]
    .sort_values(["option_type", "maturity_days", "moneyness"])
    .loc[
        :,
        [
            "row_id",
            "date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "iv_observed",
            "iv_pred",
            "pred_source",
            "is_visible_anchor",
            "is_scored_target",
        ],
    ]
    .reset_index(drop=True)
)


Fold 1 ML candidate dates


,target_date
0,2025-03-24
1,2025-03-25
2,2025-03-26
3,2025-03-27
4,2025-03-28
5,2025-03-31
6,2025-04-01
7,2025-04-02
8,2025-04-03
9,2025-04-04


Single-date leakage-safe training-bundle check


,value
target_date,2025-04-18
protocol,primary_realistic
n_history_dates,76
history_end_date,2025-04-17
n_rows_target_date,120
n_visible_anchors,52
n_scored_targets,8
n_orig_missing,60
demo_rmse,1.1341
all_scored_have_preds,True


Prediction source mix on scored targets


,count
pred_source,
structured_region_blend_callput_shrink,6
quadratic_smile_only,2


Scored target preview


,row_id,date,option_type,maturity_label,maturity_days,moneyness,iv_observed,iv_pred,pred_source,is_visible_anchor,is_scored_target
0,9128,2025-04-18,call,1M,30,0.9250,30.6785,28.6798,quadratic_smile_only,False,True
1,9150,2025-04-18,call,2M,60,0.8000,31.7153,29.8988,structured_region_blend_callput_shrink,False,True
2,9192,2025-04-18,call,3M,91,0.9750,25.8912,25.2448,structured_region_blend_callput_shrink,False,True
3,9224,2025-04-18,call,6M,182,1.0000,23.5198,24.6939,structured_region_blend_callput_shrink,False,True
4,9147,2025-04-18,put,1M,30,1.1500,24.8409,24.2783,quadratic_smile_only,False,True
5,9163,2025-04-18,put,2M,60,0.9750,27.1682,26.2605,structured_region_blend_callput_shrink,False,True
6,9187,2025-04-18,put,3M,91,0.9000,27.1568,26.9176,structured_region_blend_callput_shrink,False,True
7,9221,2025-04-18,put,6M,182,0.9500,25.3627,25.3306,structured_region_blend_callput_shrink,False,True


In [54]:

def add_true_support_columns(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()

    # Preserve the masking-stage support score separately.
    if "local_support_score" in out.columns:
        out = out.rename(columns={"local_support_score": "mask_local_support_score"})

    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    support = local_support_profile(scored_targets, visible_anchors)[
        [
            "row_id",
            "opp_option_visible",
            "same_maturity_adj_visible_count",
            "same_moneyness_adj_visible_count",
            "local_support_score",
        ]
    ].copy()

    support = support.rename(columns={"local_support_score": "true_local_support_score"})
    support["any_local_same_date_support"] = (
        support["opp_option_visible"]
        | (support["same_maturity_adj_visible_count"] > 0)
        | (support["same_moneyness_adj_visible_count"] > 0)
    )
    support["hard_case"] = ~support["any_local_same_date_support"]

    out = out.merge(support, on="row_id", how="left")

    out["opp_option_visible"] = out["opp_option_visible"].astype("boolean").fillna(False).astype(bool)
    out["same_maturity_adj_visible_count"] = out["same_maturity_adj_visible_count"].fillna(0).astype(int)
    out["same_moneyness_adj_visible_count"] = out["same_moneyness_adj_visible_count"].fillna(0).astype(int)
    out["true_local_support_score"] = out["true_local_support_score"].fillna(0).astype(int)
    out["any_local_same_date_support"] = out["any_local_same_date_support"].astype("boolean").fillna(False).astype(bool)
    out["hard_case"] = out["hard_case"].astype("boolean").fillna(False).astype(bool)

    if "mask_local_support_score" in out.columns:
        out["mask_local_support_score"] = out["mask_local_support_score"].fillna(0).astype(int)

    return out


def add_same_maturity_anchor_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    out["left_anchor_iv"] = np.nan
    out["right_anchor_iv"] = np.nan
    out["left_anchor_dist"] = np.nan
    out["right_anchor_dist"] = np.nan
    out["same_maturity_visible_anchor_count"] = 0

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        anchor_x = anchors["moneyness"].to_numpy()
        anchor_y = anchors["iv_observed"].to_numpy()
        target_x = g["moneyness"].to_numpy()

        left_iv = []
        right_iv = []
        left_dist = []
        right_dist = []

        for x in target_x:
            left_mask = anchor_x <= x
            right_mask = anchor_x >= x

            if left_mask.any():
                lx = anchor_x[left_mask][-1]
                ly = anchor_y[left_mask][-1]
                left_iv.append(ly)
                left_dist.append(abs(x - lx))
            else:
                left_iv.append(np.nan)
                left_dist.append(np.nan)

            if right_mask.any():
                rx = anchor_x[right_mask][0]
                ry = anchor_y[right_mask][0]
                right_iv.append(ry)
                right_dist.append(abs(rx - x))
            else:
                right_iv.append(np.nan)
                right_dist.append(np.nan)

        out.loc[g.index, "left_anchor_iv"] = left_iv
        out.loc[g.index, "right_anchor_iv"] = right_iv
        out.loc[g.index, "left_anchor_dist"] = left_dist
        out.loc[g.index, "right_anchor_dist"] = right_dist
        out.loc[g.index, "same_maturity_visible_anchor_count"] = len(anchors)

    return out


def add_same_node_opposite_option_feature(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    opp_visible = (
        out.loc[out["is_visible_anchor"], ["date", "moneyness", "maturity_label", "option_type", "iv_observed"]]
        .copy()
    )
    opp_visible["option_type"] = opp_visible["option_type"].map(opposite_option)
    opp_visible = opp_visible.rename(columns={"iv_observed": "opp_visible_iv_same_node"})

    out = out.merge(
        opp_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    return out


def get_nearest_moneyness_rows(df: pd.DataFrame, target: float) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    distances = (df["moneyness"] - target).abs()
    min_dist = distances.groupby(df["date"]).transform("min")
    return df.loc[min_dist.eq(distances)].copy()


def add_date_level_regime_proxy_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()
    visible = out.loc[out["is_visible_anchor"] & out["iv_observed"].notna()].copy()

    total_rows = out.groupby("date").size().rename("date_total_row_count")
    visible_count = visible.groupby("date").size().rename("date_visible_anchor_count")
    visible_ratio = (visible_count / total_rows).rename("date_visible_anchor_ratio")
    visible_mean = visible.groupby("date")["iv_observed"].mean().rename("date_visible_iv_mean")
    visible_dispersion = visible.groupby("date")["iv_observed"].std().rename("date_visible_iv_dispersion")

    atm = (
        get_nearest_moneyness_rows(visible, 1.0)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_atm_iv_proxy")
    )
    left = (
        get_nearest_moneyness_rows(visible, 0.9)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_0p9_proxy")
    )
    right = (
        get_nearest_moneyness_rows(visible, 1.1)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_1p1_proxy")
    )

    maturity_means = (
        visible.groupby(["date", "maturity_days"])["iv_observed"]
        .mean()
        .reset_index()
        .sort_values(["date", "maturity_days"])
    )
    short_end = (
        maturity_means.groupby("date").first()["iv_observed"]
        .rename("date_short_end_iv_proxy")
    )
    long_end = (
        maturity_means.groupby("date").last()["iv_observed"]
        .rename("date_long_end_iv_proxy")
    )

    proxy_df = pd.concat(
        [
            total_rows,
            visible_count,
            visible_ratio,
            visible_mean,
            visible_dispersion,
            atm,
            left,
            right,
            short_end,
            long_end,
        ],
        axis=1,
    ).reset_index()

    proxy_df["date_visible_anchor_count"] = proxy_df["date_visible_anchor_count"].fillna(0).astype(int)
    proxy_df["date_visible_anchor_ratio"] = proxy_df["date_visible_anchor_ratio"].fillna(0.0)
    proxy_df["date_skew_proxy"] = proxy_df["date_iv_0p9_proxy"] - proxy_df["date_iv_1p1_proxy"]
    proxy_df["date_term_slope_proxy"] = (
        proxy_df["date_short_end_iv_proxy"] - proxy_df["date_long_end_iv_proxy"]
    )

    out = out.merge(proxy_df, on="date", how="left")
    return out


In [55]:

def build_feature_table_for_masked_bundle(
    train_history: pd.DataFrame,
    masked_target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    base = masked_target_rows.copy()

    # Recompute all structured predictions under the same masked information set.
    linear_pred = predict_same_date_linear_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "same_date_linear_interp_pred"})

    smile_pred = predict_quadratic_smile_logm(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "smile_pred"]
    ].rename(columns={"smile_pred": "quadratic_smile_logm_pred"})

    tv_pred = predict_total_variance_maturity_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "tv_pred"]
    ].rename(columns={"tv_pred": "tv_maturity_interp_pred"})

    region_pred = predict_structured_region_blend(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred", "pred_source"]
    ].rename(
        columns={
            "iv_pred": "structured_region_blend_pred",
            "pred_source": "structured_region_blend_source",
        }
    )

    winner_pred = predict_structured_region_blend_callput_shrink(
        train_history,
        base,
        lookback_dates=lookback_dates,
        shrink_alpha=PHASE4_SHRINK_ALPHA,
    )[["row_id", "iv_pred", "pred_source"]].rename(
        columns={
            "iv_pred": "structured_winner_pred",
            "pred_source": "structured_winner_source",
        }
    )

    node_pred = predict_recent_node_median(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "recent_node_pred", "full_node_pred"]
    ]

    feat = base.merge(linear_pred, on="row_id", how="left")
    feat = feat.merge(smile_pred, on="row_id", how="left")
    feat = feat.merge(tv_pred, on="row_id", how="left")
    feat = feat.merge(region_pred, on="row_id", how="left")
    feat = feat.merge(winner_pred, on="row_id", how="left")
    feat = feat.merge(node_pred, on="row_id", how="left")

    feat = add_true_support_columns(feat)
    feat = add_same_maturity_anchor_features(feat)
    feat = add_same_node_opposite_option_feature(feat)
    feat = add_date_level_regime_proxy_features(feat)

    feat["support_score_gap"] = feat["true_local_support_score"] - feat.get("mask_local_support_score", 0)
    feat["has_opp_same_node_visible"] = feat["opp_visible_iv_same_node"].notna().astype(int)

    feat["log_moneyness"] = np.log(feat["moneyness"])
    feat["is_call"] = (feat["option_type"] == "call").astype(int)
    feat["wing_center_bucket"] = np.where(
        feat["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )
    feat["is_center"] = (feat["wing_center_bucket"] == "center").astype(int)
    feat["is_wing"] = (feat["wing_center_bucket"] == "wing").astype(int)
    feat["is_edge_maturity"] = feat["maturity_label"].isin(["1M", "6M"]).astype(int)
    feat["is_interior_maturity"] = 1 - feat["is_edge_maturity"]

    feat["smile_minus_linear"] = feat["quadratic_smile_logm_pred"] - feat["same_date_linear_interp_pred"]
    feat["tv_minus_linear"] = feat["tv_maturity_interp_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_linear"] = feat["structured_winner_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_region"] = feat["structured_winner_pred"] - feat["structured_region_blend_pred"]

    feat["target_iv"] = feat["iv_observed"]
    feat["target_residual"] = feat["target_iv"] - feat["structured_winner_pred"]

    return feat


In [56]:

def build_training_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
) -> pd.DataFrame:
    target_dates = get_ml_candidate_dates_for_fold(
        fold_row,
        min_history_dates=min_history_dates,
        n_mask_dates=n_mask_dates,
    )

    tables = []
    for target_date in target_dates:
        bundle = make_single_date_training_bundle(
            train_df,
            target_date=target_date,
            protocol_name=protocol_name,
        )

        feat = build_feature_table_for_masked_bundle(
            bundle["train_history"],
            bundle["masked_target_date"],
            lookback_dates=lookback_dates,
        )

        scored = feat.loc[feat["is_scored_target"]].copy()
        if not scored["is_orig_observed"].all():
            raise ValueError("Pseudo-masked training examples must come from originally observed rows only.")

        scored["target_date"] = pd.Timestamp(target_date)
        scored["protocol"] = protocol_name
        scored["fold"] = fold_row.fold
        scored["n_history_dates"] = bundle["train_history"]["date"].nunique()
        tables.append(scored)

    return pd.concat(tables, ignore_index=True)


def build_validation_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    val_dates = train_date_summary.loc[fold_row.val_start:fold_row.val_end].index.to_list()
    train_history = train_df.loc[train_df["date"] <= fold_row.train_end].copy()
    masked_val_rows = build_masked_validation_rows_with_protocol(
        train_df,
        val_dates,
        protocol_name=protocol_name,
    )

    feat = build_feature_table_for_masked_bundle(
        train_history,
        masked_val_rows,
        lookback_dates=lookback_dates,
    )

    scored = feat.loc[feat["is_scored_target"]].copy()
    if not scored["is_orig_observed"].all():
        raise ValueError("Validation scored rows must come from originally observed rows only.")

    scored["target_date"] = scored["date"]
    scored["protocol"] = protocol_name
    scored["fold"] = fold_row.fold
    scored["n_history_dates"] = train_history["date"].nunique()
    return scored.reset_index(drop=True)


def summarize_feature_table(feature_table: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n_rows": len(feature_table),
            "n_dates": feature_table["target_date"].nunique(),
            "date_min": feature_table["target_date"].min().date().isoformat(),
            "date_max": feature_table["target_date"].max().date().isoformat(),
            "target_iv_missing": int(feature_table["target_iv"].isna().sum()),
            "target_residual_missing": int(feature_table["target_residual"].isna().sum()),
            "winner_pred_missing": int(feature_table["structured_winner_pred"].isna().sum()),
            "mean_target_iv": feature_table["target_iv"].mean(),
            "mean_target_residual": feature_table["target_residual"].mean(),
            "rmse_structured_winner": rmse(feature_table["target_iv"], feature_table["structured_winner_pred"]),
        }
    )


def schema_alignment_report(train_table: pd.DataFrame, val_table: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    train_cols = set(train_table.columns)
    val_cols = set(val_table.columns)
    train_only = sorted(train_cols - val_cols)
    val_only = sorted(val_cols - train_cols)

    summary = pd.Series(
        {
            "n_train_cols": len(train_cols),
            "n_val_cols": len(val_cols),
            "n_common_cols": len(train_cols & val_cols),
            "n_train_only_cols": len(train_only),
            "n_val_only_cols": len(val_only),
            "schemas_match": len(train_only) == 0 and len(val_only) == 0,
        }
    )
    detail = pd.DataFrame(
        {
            "train_only_cols": pd.Series(train_only, dtype="object"),
            "val_only_cols": pd.Series(val_only, dtype="object"),
        }
    )
    return summary, detail


fold1_primary_train_table = build_training_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

fold1_primary_val_table = build_validation_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

schema_summary, schema_detail = schema_alignment_report(
    fold1_primary_train_table,
    fold1_primary_val_table,
)

print("Fold 1 / primary_realistic ML training table summary")
display(summarize_feature_table(fold1_primary_train_table).to_frame("value"))

print("Fold 1 / primary_realistic validation table summary")
display(summarize_feature_table(fold1_primary_val_table).to_frame("value"))

print("Train / validation schema alignment")
display(schema_summary.to_frame("value"))
display(schema_detail)

print("Structured winner source mix in training table")
display(
    fold1_primary_train_table["structured_winner_source"]
    .value_counts(dropna=False)
    .rename_axis("structured_winner_source")
    .to_frame("count")
)

print("Structured winner source mix in validation table")
display(
    fold1_primary_val_table["structured_winner_source"]
    .value_counts(dropna=False)
    .rename_axis("structured_winner_source")
    .to_frame("count")
)


Fold 1 / primary_realistic ML training table summary


,value
n_rows,155
n_dates,20
date_min,2025-03-24
date_max,2025-04-18
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,21.3784
mean_target_residual,0.0926
rmse_structured_winner,0.9005


Fold 1 / primary_realistic validation table summary


,value
n_rows,39
n_dates,5
date_min,2025-04-21
date_max,2025-04-25
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,19.5990
mean_target_residual,0.3840
rmse_structured_winner,1.3592


Train / validation schema alignment


...

,train_only_cols,val_only_cols


Structured winner source mix in training table


,count
structured_winner_source,
structured_region_blend_callput_shrink,93
quadratic_smile_only,35
same_date_linear_interp,15
structured_region_blend_wing,5
structured_region_blend_center,4
tv_maturity_only,3


Structured winner source mix in validation table


,count
structured_winner_source,
structured_region_blend_callput_shrink,20
quadratic_smile_only,8
same_date_linear_interp,7
tv_maturity_only,3
structured_region_blend_center,1


In [57]:

feature_cols_preview = [
    "row_id",
    "target_date",
    "option_type",
    "maturity_label",
    "moneyness",
    "target_iv",
    "structured_winner_pred",
    "target_residual",
    "same_date_linear_interp_pred",
    "quadratic_smile_logm_pred",
    "tv_maturity_interp_pred",
    "structured_region_blend_pred",
    "opp_visible_iv_same_node",
    "left_anchor_iv",
    "right_anchor_iv",
    "left_anchor_dist",
    "right_anchor_dist",
    "same_maturity_visible_anchor_count",
    "opp_option_visible",
    "same_maturity_adj_visible_count",
    "same_moneyness_adj_visible_count",
    "true_local_support_score",
    "hard_case",
    "date_atm_iv_proxy",
    "date_skew_proxy",
    "date_term_slope_proxy",
    "date_visible_anchor_ratio",
    "date_visible_iv_dispersion",
    "structured_winner_source",
]

missingness_cols = [
    "same_date_linear_interp_pred",
    "quadratic_smile_logm_pred",
    "tv_maturity_interp_pred",
    "structured_region_blend_pred",
    "structured_winner_pred",
    "opp_visible_iv_same_node",
    "left_anchor_iv",
    "right_anchor_iv",
    "left_anchor_dist",
    "right_anchor_dist",
    "date_atm_iv_proxy",
    "date_skew_proxy",
    "date_term_slope_proxy",
    "date_visible_anchor_ratio",
    "date_visible_iv_dispersion",
]

print("Training feature missingness snapshot")
display(
    fold1_primary_train_table[missingness_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .to_frame()
    .sort_values("missing_rate", ascending=False)
)

print("Validation feature missingness snapshot")
display(
    fold1_primary_val_table[missingness_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .to_frame()
    .sort_values("missing_rate", ascending=False)
)

print("Training table preview")
display(
    fold1_primary_train_table
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .loc[:, feature_cols_preview]
    .reset_index(drop=True)
    .head(30)
)

print("Validation table preview")
display(
    fold1_primary_val_table
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .loc[:, feature_cols_preview]
    .reset_index(drop=True)
    .head(30)
)


Training feature missingness snapshot


,missing_rate
tv_maturity_interp_pred,0.7290
opp_visible_iv_same_node,0.4000
quadratic_smile_logm_pred,0.2645
right_anchor_iv,0.1484
right_anchor_dist,0.1484
left_anchor_iv,0.1032
left_anchor_dist,0.1032
same_date_linear_interp_pred,0.0000
structured_region_blend_pred,0.0000
structured_winner_pred,0.0000


Validation feature missingness snapshot


,missing_rate
tv_maturity_interp_pred,0.7949
opp_visible_iv_same_node,0.4872
quadratic_smile_logm_pred,0.4359
left_anchor_iv,0.2308
left_anchor_dist,0.2308
right_anchor_iv,0.2051
right_anchor_dist,0.2051
same_date_linear_interp_pred,0.0000
structured_region_blend_pred,0.0000
structured_winner_pred,0.0000


Training table preview


,row_id,target_date,option_type,maturity_label,moneyness,target_iv,structured_winner_pred,target_residual,same_date_linear_interp_pred,quadratic_smile_logm_pred,tv_maturity_interp_pred,structured_region_blend_pred,opp_visible_iv_same_node,left_anchor_iv,right_anchor_iv,left_anchor_dist,right_anchor_dist,same_maturity_visible_anchor_count,opp_option_visible,same_maturity_adj_visible_count,same_moneyness_adj_visible_count,true_local_support_score,hard_case,date_atm_iv_proxy,date_skew_proxy,date_term_slope_proxy,date_visible_anchor_ratio,date_visible_iv_dispersion,structured_winner_source
0,6844,2025-03-24,call,1M,0.8750,24.1307,23.4565,0.6742,24.0528,23.4565,NaN,23.4565,NaN,25.3190,22.7866,0.0250,0.0250,6,False,2,0,2,False,19.3666,3.0698,3.4438,0.5250,2.3345,quadratic_smile_only
1,6874,2025-03-24,call,2M,0.8750,22.2701,22.7026,-0.4325,22.5717,22.4182,NaN,22.4182,23.5559,22.9597,22.1837,0.0250,0.0250,8,True,2,1,4,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
2,6912,2025-03-24,call,3M,0.9750,19.4499,19.0950,0.3549,19.6423,19.3881,19.0172,19.2583,18.6051,19.9420,19.3425,0.0250,0.0250,9,True,2,2,5,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
3,6944,2025-03-24,call,6M,1.0000,18.5885,18.5007,0.0878,18.0947,18.3843,NaN,18.3843,18.8498,17.6495,18.5399,0.0250,0.0250,10,True,2,1,4,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
4,6843,2025-03-24,put,1M,0.8500,26.7752,25.8167,0.9585,26.5376,25.9826,NaN,25.9826,25.3190,28.6484,24.4267,0.0500,0.0500,8,True,1,1,3,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
5,6891,2025-03-24,put,2M,1.0750,17.2588,18.6922,-1.4334,18.2079,18.2665,NaN,18.2665,19.9690,18.5981,17.8177,0.0250,0.0250,7,True,2,1,4,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
6,6907,2025-03-24,put,3M,0.9000,20.5641,20.4769,0.0872,21.5573,21.0444,20.4234,20.8270,19.4266,22.5414,18.6051,0.0250,0.0750,5,True,1,1,3,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
7,6933,2025-03-24,put,6M,0.8500,20.4437,20.1478,0.2959,20.6483,20.4125,NaN,20.4125,19.3535,21.9842,19.3123,0.0500,0.0500,10,True,1,1,3,False,19.3666,3.0698,3.4438,0.5250,2.3345,structured_region_blend_callput_shrink
8,6968,2025-03-25,call,1M,0.9250,28.6424,29.9270,-1.2846,29.8961,29.9270,NaN,29.9270,NaN,31.3400,28.4522,0.0250,0.0250,7,False,2,1,3,False,26.0735,4.1592,3.4573,0.5417,2.4341,quadratic_smile_only
9,7016,2025-03-25,call,2M,1.1500,25.7086,24.8137,0.8949,25.6030,NaN,NaN,25.6030,22.4459,25.6030,NaN,0.0250,NaN,7,True,1,1,3,False,26.0735,4.1592,3.4573,0.5417,2.4341,structured_region_blend_callput_shrink


Validation table preview


,row_id,target_date,option_type,maturity_label,moneyness,target_iv,structured_winner_pred,target_residual,same_date_linear_interp_pred,quadratic_smile_logm_pred,tv_maturity_interp_pred,structured_region_blend_pred,opp_visible_iv_same_node,left_anchor_iv,right_anchor_iv,left_anchor_dist,right_anchor_dist,same_maturity_visible_anchor_count,opp_option_visible,same_maturity_adj_visible_count,same_moneyness_adj_visible_count,true_local_support_score,hard_case,date_atm_iv_proxy,date_skew_proxy,date_term_slope_proxy,date_visible_anchor_ratio,date_visible_iv_dispersion,structured_winner_source
0,9262,2025-04-21,call,1M,1.1000,19.2304,19.1951,0.0353,19.3950,19.5234,NaN,19.5234,18.2103,19.4929,19.2972,0.0250,0.0250,9,True,2,1,4,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
1,9294,2025-04-21,call,2M,1.1250,19.0918,18.4505,0.6413,18.3872,18.5846,18.8674,18.7684,17.4968,18.8540,17.9204,0.0250,0.0250,10,True,2,2,5,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
2,9312,2025-04-21,call,3M,0.9750,18.7062,19.3098,-0.6036,19.8954,19.5126,18.8125,19.2676,19.4365,20.3649,19.6606,0.0500,0.0250,7,True,1,2,4,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
3,9344,2025-04-21,call,6M,1.0000,17.1591,18.0928,-0.9337,17.8314,18.0443,NaN,18.0443,18.2384,17.3971,18.2656,0.0250,0.0250,8,True,2,1,4,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
4,9245,2025-04-21,put,1M,0.8750,24.2840,25.1451,-0.8611,25.4055,25.1451,NaN,25.1451,NaN,26.2571,24.5538,0.0250,0.0250,8,False,2,1,3,False,19.3527,3.0547,2.1142,0.5750,2.1252,quadratic_smile_only
5,9281,2025-04-21,put,2M,0.9500,20.2648,20.5633,-0.2985,21.4680,20.9141,20.2038,20.6655,20.2566,22.2147,19.2277,0.0250,0.0750,9,True,1,2,4,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
6,9329,2025-04-21,put,3M,1.2000,16.8226,17.9854,-1.1628,17.2497,NaN,17.9854,17.9854,NaN,17.2497,NaN,0.0500,NaN,8,False,1,2,3,False,19.3527,3.0547,2.1142,0.5750,2.1252,tv_maturity_only
7,9333,2025-04-21,put,6M,0.8500,20.5319,19.8634,0.6685,19.9325,19.9058,NaN,19.9058,19.7361,21.5584,19.1196,0.0500,0.0250,10,True,2,0,3,False,19.3527,3.0547,2.1142,0.5750,2.1252,structured_region_blend_callput_shrink
8,9372,2025-04-22,call,1M,0.9750,21.0011,20.9739,0.0272,20.8571,20.9739,NaN,20.9739,NaN,21.2569,20.0576,0.0250,0.0500,6,False,1,0,1,False,18.8282,4.1716,2.8155,0.4333,2.3535,quadratic_smile_only
9,9390,2025-04-22,call,2M,0.8000,24.7991,21.2919,3.5072,21.2919,NaN,NaN,21.2919,NaN,NaN,21.2919,NaN,0.0750,5,False,0,1,1,False,18.8282,4.1716,2.8155,0.4333,2.3535,same_date_linear_interp


In [58]:

regime_cols = [
    "date_atm_iv_proxy",
    "date_skew_proxy",
    "date_term_slope_proxy",
    "date_visible_anchor_ratio",
    "date_visible_iv_dispersion",
]

phase50_sanity = pd.DataFrame(
    {
        "train_missing_rate": fold1_primary_train_table[regime_cols].isna().mean(),
        "val_missing_rate": fold1_primary_val_table[regime_cols].isna().mean(),
    }
).sort_index()

print("Phase 5.0 regime proxy coverage")
display(phase50_sanity)

print("Phase 5.0 hard-case share")
display(
    pd.DataFrame(
        {
            "train": fold1_primary_train_table["hard_case"].value_counts(normalize=True),
            "validation": fold1_primary_val_table["hard_case"].value_counts(normalize=True),
        }
    )
    .fillna(0.0)
    .rename_axis("hard_case")
)

print("Phase 5.0 sanity review complete")


Phase 5.0 regime proxy coverage


,train_missing_rate,val_missing_rate
date_atm_iv_proxy,0.0000,0.0000
date_skew_proxy,0.0000,0.0000
date_term_slope_proxy,0.0000,0.0000
date_visible_anchor_ratio,0.0000,0.0000
date_visible_iv_dispersion,0.0000,0.0000


Phase 5.0 hard-case share


,train,validation
hard_case,,
False,0.9935,0.9231
True,0.0065,0.0769


Phase 5.0 sanity review complete


In [59]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


def rmse_from_series(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


STRUCTURED_META_FEATURES = [
    "structured_winner_pred",
    "structured_region_blend_pred",
    "tv_maturity_interp_pred",
    "quadratic_smile_logm_pred",
    "same_date_linear_interp_pred",
    "opp_visible_iv_same_node",
    "true_local_support_score",
    "hard_case",
    "date_atm_iv_proxy",
    "date_skew_proxy",
    "date_term_slope_proxy",
    "date_visible_anchor_ratio",
    "date_visible_iv_dispersion",
]

FULL_RIDGE_FEATURES = [
    # Row-local
    "moneyness",
    "log_moneyness",
    "maturity_days",
    "tau",
    "is_call",
    "is_center",
    "is_wing",
    "is_edge_maturity",
    "is_interior_maturity",
    # Structured predictions
    "structured_winner_pred",
    "structured_region_blend_pred",
    "tv_maturity_interp_pred",
    "quadratic_smile_logm_pred",
    "same_date_linear_interp_pred",
    "recent_node_pred",
    "full_node_pred",
    # Structured gaps
    "smile_minus_linear",
    "tv_minus_linear",
    "winner_minus_linear",
    "winner_minus_region",
    # Same-date anchor geometry
    "opp_visible_iv_same_node",
    "has_opp_same_node_visible",
    "left_anchor_iv",
    "right_anchor_iv",
    "left_anchor_dist",
    "right_anchor_dist",
    "same_maturity_visible_anchor_count",
    # Support features
    "opp_option_visible",
    "same_maturity_adj_visible_count",
    "same_moneyness_adj_visible_count",
    "true_local_support_score",
    "mask_local_support_score",
    "support_score_gap",
    "hard_case",
    # Date-level regime proxies
    "date_atm_iv_proxy",
    "date_skew_proxy",
    "date_term_slope_proxy",
    "date_visible_anchor_ratio",
    "date_visible_iv_dispersion",
    "date_visible_iv_mean",
    "date_visible_anchor_count",
    # Categorical/context features
    "option_type",
    "maturity_label",
    "wing_center_bucket",
    "structured_winner_source",
    "structured_region_blend_source",
]


def prepare_design_matrices(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
):
    X_train = train_df.loc[:, feature_cols].copy()
    X_val = val_df.loc[:, feature_cols].copy()

    # Convert bools to ints for cleaner downstream handling.
    for df in (X_train, X_val):
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for col in bool_cols:
            df[col] = df[col].astype(int)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    # Numeric block: median imputation from train only.
    if num_cols:
        train_num = X_train[num_cols].copy()
        val_num = X_val[num_cols].copy()

        medians = train_num.median(numeric_only=True)
        train_num = train_num.fillna(medians)
        val_num = val_num.fillna(medians)
    else:
        train_num = pd.DataFrame(index=X_train.index)
        val_num = pd.DataFrame(index=X_val.index)

    # Categorical block: one-hot only if categoricals exist.
    if cat_cols:
        train_cat = pd.get_dummies(X_train[cat_cols], dummy_na=True)
        val_cat = pd.get_dummies(X_val[cat_cols], dummy_na=True)
        train_cat, val_cat = train_cat.align(val_cat, join="outer", axis=1, fill_value=0)
    else:
        train_cat = pd.DataFrame(index=X_train.index)
        val_cat = pd.DataFrame(index=X_val.index)

    X_train_final = pd.concat([train_num, train_cat], axis=1)
    X_val_final = pd.concat([val_num, val_cat], axis=1)

    return X_train_final, X_val_final



def evaluate_ridge_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    alpha: float = 1.0,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "alpha": alpha,
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[["row_id", "target_date", "target_iv", "structured_winner_pred"]].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model, X_train.columns.tolist()


In [60]:
meta_baseline_summary, meta_baseline_preds, meta_baseline_model, meta_baseline_cols = evaluate_ridge_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=STRUCTURED_META_FEATURES,
    target_col="target_residual",
    alpha=1.0,
    prediction_mode="residual",
)

ridge_direct_summary, ridge_direct_preds, ridge_direct_model, ridge_direct_cols = evaluate_ridge_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_iv",
    alpha=1.0,
    prediction_mode="direct_iv",
)

ridge_residual_summary, ridge_residual_preds, ridge_residual_model, ridge_residual_cols = evaluate_ridge_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_residual",
    alpha=1.0,
    prediction_mode="residual",
)

phase51_results = pd.DataFrame(
    [
        {"model": "structured_meta_baseline", **meta_baseline_summary},
        {"model": "ridge_direct_full", **ridge_direct_summary},
        {"model": "ridge_residual_full", **ridge_residual_summary},
    ]
).sort_values("val_rmse")

print("Phase 5.1 | Fold 1 / primary_realistic results")
display(phase51_results)


Phase 5.1 | Fold 1 / primary_realistic results


,model,alpha,target_col,prediction_mode,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
1,ridge_direct_full,1.0000,target_iv,direct_iv,65,21.3784,1.0556,1.3592
2,ridge_residual_full,1.0000,target_residual,residual,65,0.0926,1.0601,1.3592
0,structured_meta_baseline,1.0000,target_residual,residual,13,0.0926,1.2861,1.3592


In [61]:
comparison_df = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                fold1_primary_val_table["structured_winner_pred"],
            ),
        },
        {
            "model": "structured_meta_baseline",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                meta_baseline_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_residual_preds["iv_pred"],
            ),
        },
    ]
).sort_values("val_rmse")

print("Direct comparison vs structured winner")
display(comparison_df)


Direct comparison vs structured winner


,model,val_rmse
2,ridge_direct_full,1.0556
3,ridge_residual_full,1.0601
1,structured_meta_baseline,1.2861
0,structured_winner,1.3592


In [63]:
preview_compare = fold1_primary_val_table[
    [
        "row_id",
        "target_date",
        "option_type",
        "maturity_label",
        "maturity_days",
        "moneyness",
        "target_iv",
        "structured_winner_pred",
        "target_residual",
        "structured_winner_source",
    ]
].copy()

preview_compare = preview_compare.merge(
    meta_baseline_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "meta_baseline_iv_pred"}),
    on="row_id",
    how="left",
)

preview_compare = preview_compare.merge(
    ridge_direct_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_direct_iv_pred"}),
    on="row_id",
    how="left",
)

preview_compare = preview_compare.merge(
    ridge_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_residual_iv_pred"}),
    on="row_id",
    how="left",
)

print("Validation prediction preview")
display(
    preview_compare
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .reset_index(drop=True)
)


Validation prediction preview


,row_id,target_date,option_type,maturity_label,maturity_days,moneyness,target_iv,structured_winner_pred,target_residual,structured_winner_source,meta_baseline_iv_pred,ridge_direct_iv_pred,ridge_residual_iv_pred
0,9262,2025-04-21,call,1M,30,1.1000,19.2304,19.1951,0.0353,structured_region_blend_callput_shrink,19.3374,19.1179,19.1079
1,9294,2025-04-21,call,2M,60,1.1250,19.0918,18.4505,0.6413,structured_region_blend_callput_shrink,18.2991,18.3336,18.3128
2,9312,2025-04-21,call,3M,91,0.9750,18.7062,19.3098,-0.6036,structured_region_blend_callput_shrink,19.5202,19.6992,19.7011
3,9344,2025-04-21,call,6M,182,1.0000,17.1591,18.0928,-0.9337,structured_region_blend_callput_shrink,18.0478,17.9350,17.9350
4,9245,2025-04-21,put,1M,30,0.8750,24.2840,25.1451,-0.8611,quadratic_smile_only,25.9839,25.6304,25.6421
5,9281,2025-04-21,put,2M,60,0.9500,20.2648,20.5633,-0.2985,structured_region_blend_callput_shrink,21.0209,21.2988,21.3163
6,9329,2025-04-21,put,3M,91,1.2000,16.8226,17.9854,-1.1628,tv_maturity_only,17.6832,17.0630,17.0595
7,9333,2025-04-21,put,6M,182,0.8500,20.5319,19.8634,0.6685,structured_region_blend_callput_shrink,20.2158,20.8219,20.8158
8,9372,2025-04-22,call,1M,30,0.9750,21.0011,20.9739,0.0272,quadratic_smile_only,21.3921,20.2771,20.2809
9,9390,2025-04-22,call,2M,60,0.8000,24.7991,21.2919,3.5072,same_date_linear_interp,21.8117,22.5322,22.5176


In [64]:
from sklearn.ensemble import HistGradientBoostingRegressor


In [65]:
def evaluate_histgb_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    learning_rate: float = 0.05,
    max_depth: int = 3,
    max_iter: int = 300,
    min_samples_leaf: int = 10,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = HistGradientBoostingRegressor(
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_iter=max_iter,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "max_iter": max_iter,
        "min_samples_leaf": min_samples_leaf,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "target_residual",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model


In [66]:
histgb_direct_summary, histgb_direct_preds, histgb_direct_model = evaluate_histgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_iv",
    learning_rate=0.05,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    prediction_mode="direct_iv",
)

histgb_residual_summary, histgb_residual_preds, histgb_residual_model = evaluate_histgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_residual",
    learning_rate=0.05,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    prediction_mode="residual",
)

phase52_results = pd.DataFrame(
    [
        {"model": "histgb_direct_full", **histgb_direct_summary},
        {"model": "histgb_residual_full", **histgb_residual_summary},
    ]
).sort_values("val_rmse")

print("Phase 5.2 | Fold 1 / primary_realistic results")
display(phase52_results)


Phase 5.2 | Fold 1 / primary_realistic results


,model,target_col,prediction_mode,learning_rate,max_depth,max_iter,min_samples_leaf,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
1,histgb_residual_full,target_residual,residual,0.0500,3,300,10,65,0.0926,0.8914,1.3592
0,histgb_direct_full,target_iv,direct_iv,0.0500,3,300,10,65,21.3784,1.2218,1.3592


In [67]:
phase52_comparison = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                fold1_primary_val_table["structured_winner_pred"],
            ),
        },
        {
            "model": "structured_meta_baseline",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                meta_baseline_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_residual_preds["iv_pred"],
            ),
        },
        {
            "model": "histgb_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                histgb_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "histgb_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                histgb_residual_preds["iv_pred"],
            ),
        },
    ]
).sort_values("val_rmse")

print("Fold 1 / primary_realistic comparison through HistGB")
display(phase52_comparison)


Fold 1 / primary_realistic comparison through HistGB


,model,val_rmse
5,histgb_residual_full,0.8914
2,ridge_direct_full,1.0556
3,ridge_residual_full,1.0601
4,histgb_direct_full,1.2218
1,structured_meta_baseline,1.2861
0,structured_winner,1.3592


In [68]:
histgb_preview = fold1_primary_val_table[
    [
        "row_id",
        "target_date",
        "option_type",
        "maturity_label",
        "maturity_days",
        "moneyness",
        "target_iv",
        "structured_winner_pred",
        "target_residual",
        "structured_winner_source",
    ]
].copy()

histgb_preview = histgb_preview.merge(
    ridge_direct_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_direct_iv_pred"}),
    on="row_id",
    how="left",
)

histgb_preview = histgb_preview.merge(
    ridge_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_residual_iv_pred"}),
    on="row_id",
    how="left",
)

histgb_preview = histgb_preview.merge(
    histgb_direct_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "histgb_direct_iv_pred"}),
    on="row_id",
    how="left",
)

histgb_preview = histgb_preview.merge(
    histgb_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "histgb_residual_iv_pred"}),
    on="row_id",
    how="left",
)

print("Validation prediction preview through HistGB")
display(
    histgb_preview
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .reset_index(drop=True)
)


Validation prediction preview through HistGB


,row_id,target_date,option_type,maturity_label,maturity_days,moneyness,target_iv,structured_winner_pred,target_residual,structured_winner_source,ridge_direct_iv_pred,ridge_residual_iv_pred,histgb_direct_iv_pred,histgb_residual_iv_pred
0,9262,2025-04-21,call,1M,30,1.1000,19.2304,19.1951,0.0353,structured_region_blend_callput_shrink,19.1179,19.1079,19.1213,19.2284
1,9294,2025-04-21,call,2M,60,1.1250,19.0918,18.4505,0.6413,structured_region_blend_callput_shrink,18.3336,18.3128,18.6171,18.2645
2,9312,2025-04-21,call,3M,91,0.9750,18.7062,19.3098,-0.6036,structured_region_blend_callput_shrink,19.6992,19.7011,19.7626,19.3050
3,9344,2025-04-21,call,6M,182,1.0000,17.1591,18.0928,-0.9337,structured_region_blend_callput_shrink,17.9350,17.9350,17.6839,17.5824
4,9245,2025-04-21,put,1M,30,0.8750,24.2840,25.1451,-0.8611,quadratic_smile_only,25.6304,25.6421,27.0721,25.7073
5,9281,2025-04-21,put,2M,60,0.9500,20.2648,20.5633,-0.2985,structured_region_blend_callput_shrink,21.2988,21.3163,20.8887,21.0538
6,9329,2025-04-21,put,3M,91,1.2000,16.8226,17.9854,-1.1628,tv_maturity_only,17.0630,17.0595,17.0731,17.3933
7,9333,2025-04-21,put,6M,182,0.8500,20.5319,19.8634,0.6685,structured_region_blend_callput_shrink,20.8219,20.8158,21.0913,20.5784
8,9372,2025-04-22,call,1M,30,0.9750,21.0011,20.9739,0.0272,quadratic_smile_only,20.2771,20.2809,20.4925,20.9259
9,9390,2025-04-22,call,2M,60,0.8000,24.7991,21.2919,3.5072,same_date_linear_interp,22.5322,22.5176,21.8801,22.8439


In [69]:
histgb_error_compare = histgb_preview.copy()

for col in [
    "structured_winner_pred",
    "ridge_direct_iv_pred",
    "ridge_residual_iv_pred",
    "histgb_direct_iv_pred",
    "histgb_residual_iv_pred",
]:
    histgb_error_compare[f"abs_err__{col}"] = (
        histgb_error_compare["target_iv"] - histgb_error_compare[col]
    ).abs()

print("Mean absolute error comparison on validation rows")
display(
    histgb_error_compare[
        [c for c in histgb_error_compare.columns if c.startswith("abs_err__")]
    ]
    .mean()
    .rename("mae")
    .to_frame()
    .sort_values("mae")
)


Mean absolute error comparison on validation rows


,mae
abs_err__histgb_residual_iv_pred,0.7026
abs_err__ridge_direct_iv_pred,0.8134
abs_err__ridge_residual_iv_pred,0.8179
abs_err__histgb_direct_iv_pred,0.8374
abs_err__structured_winner_pred,0.9062


In [70]:
from xgboost import XGBRegressor


In [71]:
def evaluate_xgb_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    prediction_mode: str = "direct_iv",
    n_estimators: int = 300,
    learning_rate: float = 0.05,
    max_depth: int = 3,
    min_child_weight: int = 5,
    subsample: float = 0.90,
    colsample_bytree: float = 0.80,
    reg_lambda: float = 1.0,
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda,
        random_state=42,
        n_jobs=4,
        tree_method="hist",
    )

    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "reg_lambda": reg_lambda,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "target_residual",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model, X_train.columns.tolist()


In [72]:
xgb_direct_summary, xgb_direct_preds, xgb_direct_model, xgb_direct_cols = evaluate_xgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_iv",
    prediction_mode="direct_iv",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    subsample=0.90,
    colsample_bytree=0.80,
    reg_lambda=1.0,
)

xgb_residual_summary, xgb_residual_preds, xgb_residual_model, xgb_residual_cols = evaluate_xgb_model(
    fold1_primary_train_table,
    fold1_primary_val_table,
    feature_cols=FULL_RIDGE_FEATURES,
    target_col="target_residual",
    prediction_mode="residual",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    subsample=0.90,
    colsample_bytree=0.80,
    reg_lambda=1.0,
)

phase53_results = pd.DataFrame(
    [
        {"model": "xgb_direct_full", **xgb_direct_summary},
        {"model": "xgb_residual_full", **xgb_residual_summary},
    ]
).sort_values("val_rmse")

print("Phase 5.3 | Fold 1 / primary_realistic results")
display(phase53_results)


Phase 5.3 | Fold 1 / primary_realistic results


,model,target_col,prediction_mode,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
1,xgb_residual_full,target_residual,residual,300,0.0500,3,5,0.9000,0.8000,1.0000,65,0.0926,0.9131,1.3592
0,xgb_direct_full,target_iv,direct_iv,300,0.0500,3,5,0.9000,0.8000,1.0000,65,21.3784,1.1661,1.3592


In [73]:
phase53_comparison = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                fold1_primary_val_table["structured_winner_pred"],
            ),
        },
        {
            "model": "structured_meta_baseline",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                meta_baseline_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "ridge_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                ridge_residual_preds["iv_pred"],
            ),
        },
        {
            "model": "histgb_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                histgb_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "histgb_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                histgb_residual_preds["iv_pred"],
            ),
        },
        {
            "model": "xgb_direct_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                xgb_direct_preds["iv_pred"],
            ),
        },
        {
            "model": "xgb_residual_full",
            "val_rmse": rmse_from_series(
                fold1_primary_val_table["target_iv"],
                xgb_residual_preds["iv_pred"],
            ),
        },
    ]
).sort_values("val_rmse")

print("Fold 1 / primary_realistic comparison through XGBoost")
display(phase53_comparison)


Fold 1 / primary_realistic comparison through XGBoost


,model,val_rmse
5,histgb_residual_full,0.8914
7,xgb_residual_full,0.9131
2,ridge_direct_full,1.0556
3,ridge_residual_full,1.0601
6,xgb_direct_full,1.1661
4,histgb_direct_full,1.2218
1,structured_meta_baseline,1.2861
0,structured_winner,1.3592


In [74]:
xgb_preview = fold1_primary_val_table[
    [
        "row_id",
        "target_date",
        "option_type",
        "maturity_label",
        "maturity_days",
        "moneyness",
        "target_iv",
        "structured_winner_pred",
        "target_residual",
        "structured_winner_source",
    ]
].copy()

xgb_preview = xgb_preview.merge(
    ridge_direct_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_direct_iv_pred"}),
    on="row_id",
    how="left",
)

xgb_preview = xgb_preview.merge(
    ridge_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "ridge_residual_iv_pred"}),
    on="row_id",
    how="left",
)

xgb_preview = xgb_preview.merge(
    histgb_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "histgb_residual_iv_pred"}),
    on="row_id",
    how="left",
)

xgb_preview = xgb_preview.merge(
    xgb_direct_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "xgb_direct_iv_pred"}),
    on="row_id",
    how="left",
)

xgb_preview = xgb_preview.merge(
    xgb_residual_preds[["row_id", "iv_pred"]].rename(columns={"iv_pred": "xgb_residual_iv_pred"}),
    on="row_id",
    how="left",
)

print("Validation prediction preview through XGBoost")
display(
    xgb_preview
    .sort_values(["target_date", "option_type", "maturity_days", "moneyness"])
    .reset_index(drop=True)
)


Validation prediction preview through XGBoost


,row_id,target_date,option_type,maturity_label,maturity_days,moneyness,target_iv,structured_winner_pred,target_residual,structured_winner_source,ridge_direct_iv_pred,ridge_residual_iv_pred,histgb_residual_iv_pred,xgb_direct_iv_pred,xgb_residual_iv_pred
0,9262,2025-04-21,call,1M,30,1.1000,19.2304,19.1951,0.0353,structured_region_blend_callput_shrink,19.1179,19.1079,19.2284,19.4002,19.5264
1,9294,2025-04-21,call,2M,60,1.1250,19.0918,18.4505,0.6413,structured_region_blend_callput_shrink,18.3336,18.3128,18.2645,18.5591,18.4055
2,9312,2025-04-21,call,3M,91,0.9750,18.7062,19.3098,-0.6036,structured_region_blend_callput_shrink,19.6992,19.7011,19.3050,19.6103,19.2800
3,9344,2025-04-21,call,6M,182,1.0000,17.1591,18.0928,-0.9337,structured_region_blend_callput_shrink,17.9350,17.9350,17.5824,17.7791,17.8502
4,9245,2025-04-21,put,1M,30,0.8750,24.2840,25.1451,-0.8611,quadratic_smile_only,25.6304,25.6421,25.7073,26.0929,25.0755
5,9281,2025-04-21,put,2M,60,0.9500,20.2648,20.5633,-0.2985,structured_region_blend_callput_shrink,21.2988,21.3163,21.0538,20.8075,20.6962
6,9329,2025-04-21,put,3M,91,1.2000,16.8226,17.9854,-1.1628,tv_maturity_only,17.0630,17.0595,17.3933,17.0841,17.5148
7,9333,2025-04-21,put,6M,182,0.8500,20.5319,19.8634,0.6685,structured_region_blend_callput_shrink,20.8219,20.8158,20.5784,20.8835,20.4724
8,9372,2025-04-22,call,1M,30,0.9750,21.0011,20.9739,0.0272,quadratic_smile_only,20.2771,20.2809,20.9259,20.2848,20.6388
9,9390,2025-04-22,call,2M,60,0.8000,24.7991,21.2919,3.5072,same_date_linear_interp,22.5322,22.5176,22.8439,21.2821,22.5348


In [75]:
xgb_residual_importance = pd.DataFrame(
    {
        "feature": xgb_residual_cols,
        "importance": xgb_residual_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

print("XGBoost residual | top feature importances")
display(xgb_residual_importance.head(20))


XGBoost residual | top feature importances


,feature,importance
15,full_node_pred,0.0946
51,structured_region_blend_source_same_date_linea...,0.0722
0,moneyness,0.0717
31,mask_local_support_score,0.0674
21,has_opp_same_node_visible,0.0426
19,winner_minus_region,0.0329
38,date_visible_iv_dispersion,0.0309
23,right_anchor_iv,0.0287
36,date_term_slope_proxy,0.0270
40,date_visible_anchor_count,0.0267


In [76]:
PROTOCOLS = ["primary_realistic", "stress_test"]


def run_structured_winner_baseline(train_table: pd.DataFrame, val_table: pd.DataFrame):
    pred_df = val_table[
        [
            "row_id",
            "target_date",
            "target_iv",
            "structured_winner_pred",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["iv_pred"] = pred_df["structured_winner_pred"]

    summary = {
        "model": "structured_winner",
        "target_col": "target_iv",
        "prediction_mode": "direct_iv",
        "val_rmse": rmse_from_series(val_table["target_iv"], pred_df["iv_pred"]),
        "n_features_after_encoding": 0,
    }
    return summary, pred_df


def run_ridge_direct_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model, cols = evaluate_ridge_model(
        train_table,
        val_table,
        feature_cols=FULL_RIDGE_FEATURES,
        target_col="target_iv",
        alpha=1.0,
        prediction_mode="direct_iv",
    )
    summary = {"model": "ridge_direct_full", **summary}
    return summary, pred_df


def run_histgb_residual_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model = evaluate_histgb_model(
        train_table,
        val_table,
        feature_cols=FULL_RIDGE_FEATURES,
        target_col="target_residual",
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        prediction_mode="residual",
    )
    summary = {"model": "histgb_residual_full", **summary}
    return summary, pred_df


def run_xgb_residual_full(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model, cols = evaluate_xgb_model(
        train_table,
        val_table,
        feature_cols=FULL_RIDGE_FEATURES,
        target_col="target_residual",
        prediction_mode="residual",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        subsample=0.90,
        colsample_bytree=0.80,
        reg_lambda=1.0,
    )
    summary = {"model": "xgb_residual_full", **summary}
    return summary, pred_df


MODEL_RUNNERS = {
    "structured_winner": run_structured_winner_baseline,
    "ridge_direct_full": run_ridge_direct_full,
    "histgb_residual_full": run_histgb_residual_full,
    "xgb_residual_full": run_xgb_residual_full,
}


In [77]:
table_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = build_training_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        val_table = build_validation_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        table_cache[(protocol_name, fold_id)] = {
            "train": train_table,
            "val": val_table,
        }

print("Cached fold/protocol tables")
display(
    pd.DataFrame(
        [
            {
                "protocol": protocol_name,
                "fold": fold_id,
                "n_train_rows": len(payload["train"]),
                "n_train_dates": payload["train"]["target_date"].nunique(),
                "n_val_rows": len(payload["val"]),
                "n_val_dates": payload["val"]["target_date"].nunique(),
                "schemas_match": set(payload["train"].columns) == set(payload["val"].columns),
            }
            for (protocol_name, fold_id), payload in table_cache.items()
        ]
    ).sort_values(["protocol", "fold"]).reset_index(drop=True)
)


Cached fold/protocol tables


,protocol,fold,n_train_rows,n_train_dates,n_val_rows,n_val_dates,schemas_match
0,primary_realistic,1,155,20,39,5,True
1,primary_realistic,2,155,20,38,5,True
2,primary_realistic,3,155,20,39,5,True
3,primary_realistic,4,156,20,40,5,True
4,stress_test,1,155,20,39,5,True
5,stress_test,2,155,20,38,5,True
6,stress_test,3,155,20,39,5,True
7,stress_test,4,156,20,40,5,True


In [78]:
multi_fold_rows = []
prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])
        train_table = table_cache[(protocol_name, fold_id)]["train"]
        val_table = table_cache[(protocol_name, fold_id)]["val"]

        for model_name, runner in MODEL_RUNNERS.items():
            summary, pred_df = runner(train_table, val_table)

            summary_row = {
                "protocol": protocol_name,
                "fold": fold_id,
                "n_train_rows": len(train_table),
                "n_val_rows": len(val_table),
                "structured_winner_rmse": rmse_from_series(
                    val_table["target_iv"],
                    val_table["structured_winner_pred"],
                ),
                **summary,
            }
            multi_fold_rows.append(summary_row)

            pred_df = pred_df.copy()
            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

multi_fold_results = pd.DataFrame(multi_fold_rows).sort_values(
    ["protocol", "fold", "val_rmse", "model"]
).reset_index(drop=True)

print("Multi-fold / multi-protocol model results")
display(multi_fold_results)


Multi-fold / multi-protocol model results


,protocol,fold,n_train_rows,n_val_rows,structured_winner_rmse,model,target_col,prediction_mode,val_rmse,n_features_after_encoding,alpha,train_target_mean,learning_rate,max_depth,max_iter,min_samples_leaf,n_estimators,min_child_weight,subsample,colsample_bytree,reg_lambda
0,primary_realistic,1,155,39,1.3592,histgb_residual_full,target_residual,residual,0.8914,65,NaN,0.0926,0.0500,3.0000,300.0000,10.0000,NaN,NaN,NaN,NaN,NaN
1,primary_realistic,1,155,39,1.3592,xgb_residual_full,target_residual,residual,0.9131,65,NaN,0.0926,0.0500,3.0000,NaN,NaN,300.0000,5.0000,0.9000,0.8000,1.0000
2,primary_realistic,1,155,39,1.3592,ridge_direct_full,target_iv,direct_iv,1.0556,65,1.0000,21.3784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,primary_realistic,1,155,39,1.3592,structured_winner,target_iv,direct_iv,1.3592,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,primary_realistic,2,155,38,1.0373,histgb_residual_full,target_residual,residual,0.8928,65,NaN,0.1957,0.0500,3.0000,300.0000,10.0000,NaN,NaN,NaN,NaN,NaN
5,primary_realistic,2,155,38,1.0373,xgb_residual_full,target_residual,residual,0.9292,65,NaN,0.1957,0.0500,3.0000,NaN,NaN,300.0000,5.0000,0.9000,0.8000,1.0000
6,primary_realistic,2,155,38,1.0373,ridge_direct_full,target_iv,direct_iv,0.9294,65,1.0000,20.7935,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,primary_realistic,2,155,38,1.0373,structured_winner,target_iv,direct_iv,1.0373,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,primary_realistic,3,155,39,0.7840,structured_winner,target_iv,direct_iv,0.7840,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,primary_realistic,3,155,39,0.7840,xgb_residual_full,target_residual,residual,0.8113,65,NaN,0.1308,0.0500,3.0000,NaN,NaN,300.0000,5.0000,0.9000,0.8000,1.0000


In [79]:
protocol_summary = (
    multi_fold_results
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
        mean_n_val_rows=("n_val_rows", "mean"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Protocol-level summary")
display(protocol_summary)


Protocol-level summary


,protocol,model,mean_rmse,min_rmse,max_rmse,mean_n_val_rows
0,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928,39.0000
3,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292,39.0000
1,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,39.0000
2,primary_realistic,structured_winner,1.0638,0.7840,1.3592,39.0000
5,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,39.0000
7,stress_test,xgb_residual_full,0.9224,0.8464,1.0579,39.0000
4,stress_test,histgb_residual_full,0.9415,0.8590,1.1233,39.0000
6,stress_test,structured_winner,1.1847,1.0614,1.3163,39.0000


In [80]:
protocol_pivot = (
    protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

print("Protocol comparison pivot")
display(protocol_pivot)


Protocol comparison pivot


protocol,model,primary_realistic,stress_test
0,histgb_residual_full,0.8724,0.9415
1,ridge_direct_full,0.9245,0.8463
2,structured_winner,1.0638,1.1847
3,xgb_residual_full,0.8756,0.9224


In [81]:
structured_lookup = (
    protocol_summary.loc[protocol_summary["model"] == "structured_winner", ["protocol", "mean_rmse"]]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

protocol_gain_table = (
    protocol_summary.merge(structured_lookup, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Gain/loss vs structured winner by protocol")
display(protocol_gain_table)


Gain/loss vs structured winner by protocol


,protocol,model,mean_rmse,min_rmse,max_rmse,mean_n_val_rows,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928,39.0000,1.0638,-0.1915
1,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292,39.0000,1.0638,-0.1882
2,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,39.0000,1.0638,-0.1393
3,primary_realistic,structured_winner,1.0638,0.7840,1.3592,39.0000,1.0638,0.0000
4,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,39.0000,1.1847,-0.3384
5,stress_test,xgb_residual_full,0.9224,0.8464,1.0579,39.0000,1.1847,-0.2623
6,stress_test,histgb_residual_full,0.9415,0.8590,1.1233,39.0000,1.1847,-0.2432
7,stress_test,structured_winner,1.1847,1.0614,1.3163,39.0000,1.1847,0.0000


In [82]:
fold_rank_view = (
    multi_fold_results[
        [
            "protocol",
            "fold",
            "model",
            "val_rmse",
        ]
    ]
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Fold-level ranking view")
display(fold_rank_view)


Fold-level ranking view


,protocol,fold,model,val_rmse
0,primary_realistic,1,histgb_residual_full,0.8914
1,primary_realistic,1,xgb_residual_full,0.9131
2,primary_realistic,1,ridge_direct_full,1.0556
3,primary_realistic,1,structured_winner,1.3592
4,primary_realistic,2,histgb_residual_full,0.8928
5,primary_realistic,2,xgb_residual_full,0.9292
6,primary_realistic,2,ridge_direct_full,0.9294
7,primary_realistic,2,structured_winner,1.0373
8,primary_realistic,3,structured_winner,0.7840
9,primary_realistic,3,xgb_residual_full,0.8113


In [83]:
FRONTLINE_MODELS = [
    "structured_winner",
    "ridge_direct_full",
    "histgb_residual_full",
    "xgb_residual_full",
]


def build_frontline_diagnostics_df(
    prediction_cache: dict,
    table_cache: dict,
    model_names=None,
) -> pd.DataFrame:
    model_names = model_names or FRONTLINE_MODELS
    rows = []

    for (protocol_name, fold_id, model_name), pred_df in prediction_cache.items():
        if model_name not in model_names:
            continue

        val_table = table_cache[(protocol_name, fold_id)]["val"].copy()

        slice_cols = [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "wing_center_bucket",
            "hard_case",
            "structured_winner_source",
            "true_local_support_score",
            "same_maturity_visible_anchor_count",
            "opp_option_visible",
        ]

        add_cols = [c for c in slice_cols if c not in pred_df.columns]

        enriched = pred_df.copy().merge(
            val_table[["row_id", *add_cols]],
            on="row_id",
            how="left",
        )

        enriched["protocol"] = protocol_name
        enriched["fold"] = fold_id
        enriched["model"] = model_name
        enriched["hard_case_bucket"] = np.where(
            enriched["hard_case"],
            "hard_case",
            "non_hard_case",
        )
        enriched["abs_error"] = (enriched["target_iv"] - enriched["iv_pred"]).abs()
        enriched["sq_error"] = (enriched["target_iv"] - enriched["iv_pred"]) ** 2

        rows.append(enriched)

    return pd.concat(rows, ignore_index=True)


def slice_error_table(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)
    base_cols = ["protocol", "model", *group_cols]

    out = (
        df.groupby(base_cols, dropna=False)
        .agg(
            n=("row_id", "size"),
            mae=("abs_error", "mean"),
            mean_sq_error=("sq_error", "mean"),
        )
        .reset_index()
    )

    out["rmse"] = np.sqrt(out["mean_sq_error"])
    out = out.drop(columns=["mean_sq_error"])

    return out.sort_values(base_cols + ["rmse"]).reset_index(drop=True)


def delta_vs_structured(slice_table: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)

    pivot = (
        slice_table.pivot_table(
            index=["protocol", *group_cols],
            columns="model",
            values="rmse",
        )
        .reset_index()
    )

    for model_name in FRONTLINE_MODELS:
        if model_name == "structured_winner":
            continue
        if model_name in pivot.columns:
            pivot[f"{model_name}__delta_vs_structured"] = (
                pivot[model_name] - pivot["structured_winner"]
            )

    return pivot


In [84]:
frontline_diag_df = build_frontline_diagnostics_df(
    prediction_cache=prediction_cache,
    table_cache=table_cache,
    model_names=FRONTLINE_MODELS,
)

overall_diag = slice_error_table(frontline_diag_df, group_cols=[])

print("Overall diagnostic sanity check")
display(overall_diag)


Overall diagnostic sanity check


,protocol,model,n,mae,rmse
0,primary_realistic,histgb_residual_full,156,0.7039,0.8724
1,primary_realistic,ridge_direct_full,156,0.7109,0.9276
2,primary_realistic,structured_winner,156,0.7831,1.0834
3,primary_realistic,xgb_residual_full,156,0.6839,0.8764
4,stress_test,histgb_residual_full,156,0.7638,0.9475
5,stress_test,ridge_direct_full,156,0.6870,0.8542
6,stress_test,structured_winner,156,0.8722,1.1883
7,stress_test,xgb_residual_full,156,0.7445,0.9256


In [85]:
hard_case_diag = slice_error_table(
    frontline_diag_df,
    group_cols=["hard_case_bucket"],
)

hard_case_delta = delta_vs_structured(
    hard_case_diag,
    group_cols=["hard_case_bucket"],
)

print("Hard-case vs non-hard-case RMSE")
display(hard_case_diag)

print("Hard-case vs non-hard-case delta vs structured winner")
display(hard_case_delta)


Hard-case vs non-hard-case RMSE


,protocol,model,hard_case_bucket,n,mae,rmse
0,primary_realistic,histgb_residual_full,hard_case,6,1.0209,1.3506
1,primary_realistic,histgb_residual_full,non_hard_case,150,0.6912,0.8477
2,primary_realistic,ridge_direct_full,hard_case,6,1.6564,2.0690
3,primary_realistic,ridge_direct_full,non_hard_case,150,0.6731,0.8507
4,primary_realistic,structured_winner,hard_case,6,1.6879,2.3840
5,primary_realistic,structured_winner,non_hard_case,150,0.7469,0.9967
6,primary_realistic,xgb_residual_full,hard_case,6,1.1778,1.5262
7,primary_realistic,xgb_residual_full,non_hard_case,150,0.6641,0.8400
8,stress_test,histgb_residual_full,hard_case,12,1.0102,1.3274
9,stress_test,histgb_residual_full,non_hard_case,144,0.7432,0.9087


Hard-case vs non-hard-case delta vs structured winner


model,protocol,hard_case_bucket,histgb_residual_full,ridge_direct_full,structured_winner,xgb_residual_full,ridge_direct_full__delta_vs_structured,histgb_residual_full__delta_vs_structured,xgb_residual_full__delta_vs_structured
0,primary_realistic,hard_case,1.3506,2.0690,2.3840,1.5262,-0.3149,-1.0334,-0.8578
1,primary_realistic,non_hard_case,0.8477,0.8507,0.9967,0.8400,-0.1460,-0.1490,-0.1567
2,stress_test,hard_case,1.3274,1.2178,1.3694,1.3341,-0.1515,-0.0420,-0.0352
3,stress_test,non_hard_case,0.9087,0.8167,1.1719,0.8831,-0.3553,-0.2632,-0.2888


In [86]:
wing_center_diag = slice_error_table(
    frontline_diag_df,
    group_cols=["wing_center_bucket"],
)

wing_center_delta = delta_vs_structured(
    wing_center_diag,
    group_cols=["wing_center_bucket"],
)

print("Wing vs center RMSE")
display(wing_center_diag)

print("Wing vs center delta vs structured winner")
display(wing_center_delta)


Wing vs center RMSE


,protocol,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,histgb_residual_full,center,65,0.5474,0.6967
1,primary_realistic,histgb_residual_full,wing,91,0.8157,0.9788
2,primary_realistic,ridge_direct_full,center,65,0.6021,0.7847
3,primary_realistic,ridge_direct_full,wing,91,0.7887,1.0174
4,primary_realistic,structured_winner,center,65,0.5069,0.6537
5,primary_realistic,structured_winner,wing,91,0.9804,1.3065
6,primary_realistic,xgb_residual_full,center,65,0.5382,0.6866
7,primary_realistic,xgb_residual_full,wing,91,0.7879,0.9899
8,stress_test,histgb_residual_full,center,49,0.7144,0.8272
9,stress_test,histgb_residual_full,wing,107,0.7864,0.9978


Wing vs center delta vs structured winner


model,protocol,wing_center_bucket,histgb_residual_full,ridge_direct_full,structured_winner,xgb_residual_full,ridge_direct_full__delta_vs_structured,histgb_residual_full__delta_vs_structured,xgb_residual_full__delta_vs_structured
0,primary_realistic,center,0.6967,0.7847,0.6537,0.6866,0.1310,0.0430,0.0329
1,primary_realistic,wing,0.9788,1.0174,1.3065,0.9899,-0.2891,-0.3277,-0.3166
2,stress_test,center,0.8272,0.7958,0.6479,0.8014,0.1479,0.1793,0.1535
3,stress_test,wing,0.9978,0.8797,1.3662,0.9772,-0.4864,-0.3683,-0.3889


In [87]:
maturity_diag = slice_error_table(
    frontline_diag_df,
    group_cols=["maturity_label"],
)

maturity_delta = delta_vs_structured(
    maturity_diag,
    group_cols=["maturity_label"],
)

print("Maturity RMSE")
display(maturity_diag)

print("Maturity delta vs structured winner")
display(maturity_delta)


Maturity RMSE


,protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,histgb_residual_full,1M,40,0.7598,0.9602
1,primary_realistic,histgb_residual_full,2M,39,0.7208,0.9213
2,primary_realistic,histgb_residual_full,3M,39,0.6829,0.7923
3,primary_realistic,histgb_residual_full,6M,38,0.6493,0.7994
4,primary_realistic,ridge_direct_full,1M,40,0.8074,1.0851
5,primary_realistic,ridge_direct_full,2M,39,0.6356,0.9238
6,primary_realistic,ridge_direct_full,3M,39,0.7603,0.8905
7,primary_realistic,ridge_direct_full,6M,38,0.6360,0.7767
8,primary_realistic,structured_winner,1M,40,0.7980,1.0827
9,primary_realistic,structured_winner,2M,39,0.8972,1.4125


Maturity delta vs structured winner


model,protocol,maturity_label,histgb_residual_full,ridge_direct_full,structured_winner,xgb_residual_full,ridge_direct_full__delta_vs_structured,histgb_residual_full__delta_vs_structured,xgb_residual_full__delta_vs_structured
0,primary_realistic,1M,0.9602,1.0851,1.0827,0.9656,0.0024,-0.1226,-0.1171
1,primary_realistic,2M,0.9213,0.9238,1.4125,0.9431,-0.4888,-0.4912,-0.4694
2,primary_realistic,3M,0.7923,0.8905,0.8437,0.8030,0.0468,-0.0514,-0.0407
3,primary_realistic,6M,0.7994,0.7767,0.8980,0.7726,-0.1213,-0.0986,-0.1254
4,stress_test,1M,0.8153,0.8091,1.1906,0.7764,-0.3814,-0.3753,-0.4142
5,stress_test,2M,1.0768,0.9070,1.6369,1.0364,-0.7299,-0.5601,-0.6005
6,stress_test,3M,0.8194,0.7974,0.8410,0.8293,-0.0435,-0.0216,-0.0116
7,stress_test,6M,1.0521,0.8998,0.9104,1.0365,-0.0106,0.1417,0.1261


In [88]:
source_diag = slice_error_table(
    frontline_diag_df,
    group_cols=["structured_winner_source"],
)

source_delta = delta_vs_structured(
    source_diag,
    group_cols=["structured_winner_source"],
)

print("Structured winner source RMSE")
display(source_diag)

print("Structured winner source delta vs structured winner")
display(source_delta)


Structured winner source RMSE


,protocol,model,structured_winner_source,n,mae,rmse
0,primary_realistic,histgb_residual_full,quadratic_smile_only,34,0.7771,0.9239
1,primary_realistic,histgb_residual_full,same_date_linear_interp,18,0.8294,1.1380
2,primary_realistic,histgb_residual_full,structured_region_blend_callput_shrink,89,0.6639,0.8021
3,primary_realistic,histgb_residual_full,structured_region_blend_center,4,0.4742,0.6751
4,primary_realistic,histgb_residual_full,structured_region_blend_wing,1,0.4038,0.4038
5,primary_realistic,histgb_residual_full,tv_maturity_only,10,0.7071,0.8467
6,primary_realistic,ridge_direct_full,quadratic_smile_only,34,0.6762,0.8386
7,primary_realistic,ridge_direct_full,same_date_linear_interp,18,1.1429,1.5341
8,primary_realistic,ridge_direct_full,structured_region_blend_callput_shrink,89,0.6629,0.8300
9,primary_realistic,ridge_direct_full,structured_region_blend_center,4,0.3885,0.4109


Structured winner source delta vs structured winner


model,protocol,structured_winner_source,histgb_residual_full,ridge_direct_full,structured_winner,xgb_residual_full,ridge_direct_full__delta_vs_structured,histgb_residual_full__delta_vs_structured,xgb_residual_full__delta_vs_structured
0,primary_realistic,quadratic_smile_only,0.9239,0.8386,0.7944,0.8777,0.0443,0.1295,0.0834
1,primary_realistic,same_date_linear_interp,1.1380,1.5341,1.8214,1.2145,-0.2873,-0.6834,-0.6068
2,primary_realistic,structured_region_blend_callput_shrink,0.8021,0.8300,1.0227,0.8052,-0.1926,-0.2206,-0.2175
3,primary_realistic,structured_region_blend_center,0.6751,0.4109,0.4273,0.6755,-0.0165,0.2478,0.2482
4,primary_realistic,structured_region_blend_wing,0.4038,0.0561,0.2507,0.3871,-0.1946,0.1532,0.1364
5,primary_realistic,tv_maturity_only,0.8467,0.7719,0.8983,0.8594,-0.1264,-0.0516,-0.0389
6,stress_test,quadratic_smile_only,0.8894,0.8033,0.6615,0.8813,0.1418,0.2279,0.2198
7,stress_test,same_date_linear_interp,0.9580,0.8776,1.5004,0.9362,-0.6229,-0.5424,-0.5642
8,stress_test,structured_region_blend_callput_shrink,0.9199,0.9041,1.3330,0.8745,-0.4288,-0.4131,-0.4585
9,stress_test,structured_region_blend_center,0.7529,0.5667,0.4182,0.8546,0.1485,0.3347,0.4364


In [89]:
def winner_by_slice(slice_table: pd.DataFrame, slice_col: str) -> pd.DataFrame:
    return (
        slice_table.sort_values(["protocol", slice_col, "rmse"])
        .groupby(["protocol", slice_col], as_index=False)
        .first()[["protocol", slice_col, "model", "rmse", "n"]]
        .rename(columns={"model": "best_model", "rmse": "best_rmse"})
    )


print("Best model by hard-case bucket")
display(winner_by_slice(hard_case_diag, "hard_case_bucket"))

print("Best model by wing/center")
display(winner_by_slice(wing_center_diag, "wing_center_bucket"))

print("Best model by maturity")
display(winner_by_slice(maturity_diag, "maturity_label"))


Best model by hard-case bucket


,protocol,hard_case_bucket,best_model,best_rmse,n
0,primary_realistic,hard_case,histgb_residual_full,1.3506,6
1,primary_realistic,non_hard_case,xgb_residual_full,0.8400,150
2,stress_test,hard_case,ridge_direct_full,1.2178,12
3,stress_test,non_hard_case,ridge_direct_full,0.8167,144


Best model by wing/center


,protocol,wing_center_bucket,best_model,best_rmse,n
0,primary_realistic,center,structured_winner,0.6537,65
1,primary_realistic,wing,histgb_residual_full,0.9788,91
2,stress_test,center,structured_winner,0.6479,49
3,stress_test,wing,ridge_direct_full,0.8797,107


Best model by maturity


,protocol,maturity_label,best_model,best_rmse,n
0,primary_realistic,1M,histgb_residual_full,0.9602,40
1,primary_realistic,2M,histgb_residual_full,0.9213,39
2,primary_realistic,3M,histgb_residual_full,0.7923,39
3,primary_realistic,6M,xgb_residual_full,0.7726,38
4,stress_test,1M,xgb_residual_full,0.7764,40
5,stress_test,2M,ridge_direct_full,0.9070,39
6,stress_test,3M,ridge_direct_full,0.7974,39
7,stress_test,6M,ridge_direct_full,0.8998,38


In [90]:
HYBRID_BASE_MODELS = [
    "structured_winner",
    "ridge_direct_full",
    "histgb_residual_full",
]


def build_prediction_panel_for_fold_protocol(
    protocol_name: str,
    fold_id: int,
    table_cache: dict,
    prediction_cache: dict,
    model_names=None,
) -> pd.DataFrame:
    model_names = model_names or HYBRID_BASE_MODELS

    base = table_cache[(protocol_name, fold_id)]["val"][
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "structured_winner_source",
            "wing_center_bucket",
            "hard_case",
            "true_local_support_score",
        ]
    ].copy()

    for model_name in model_names:
        pred_col = f"{model_name}__iv_pred"
        pred_df = prediction_cache[(protocol_name, fold_id, model_name)][["row_id", "iv_pred"]].rename(
            columns={"iv_pred": pred_col}
        )
        base = base.merge(pred_df, on="row_id", how="left")

    return base


panel_example = build_prediction_panel_for_fold_protocol(
    "primary_realistic",
    1,
    table_cache=table_cache,
    prediction_cache=prediction_cache,
)

print("Prediction panel example")
display(panel_example.head(10))


Prediction panel example


,row_id,target_date,option_type,maturity_label,maturity_days,moneyness,target_iv,structured_winner_pred,structured_winner_source,wing_center_bucket,hard_case,true_local_support_score,structured_winner__iv_pred,ridge_direct_full__iv_pred,histgb_residual_full__iv_pred
0,9262,2025-04-21,call,1M,30,1.1000,19.2304,19.1951,structured_region_blend_callput_shrink,center,False,4,19.1951,19.1179,19.2284
1,9245,2025-04-21,put,1M,30,0.8750,24.2840,25.1451,quadratic_smile_only,wing,False,3,25.1451,25.6304,25.7073
2,9294,2025-04-21,call,2M,60,1.1250,19.0918,18.4505,structured_region_blend_callput_shrink,wing,False,5,18.4505,18.3336,18.2645
3,9281,2025-04-21,put,2M,60,0.9500,20.2648,20.5633,structured_region_blend_callput_shrink,center,False,4,20.5633,21.2988,21.0538
4,9312,2025-04-21,call,3M,91,0.9750,18.7062,19.3098,structured_region_blend_callput_shrink,center,False,4,19.3098,19.6992,19.3050
5,9329,2025-04-21,put,3M,91,1.2000,16.8226,17.9854,tv_maturity_only,wing,False,3,17.9854,17.0630,17.3933
6,9344,2025-04-21,call,6M,182,1.0000,17.1591,18.0928,structured_region_blend_callput_shrink,center,False,4,18.0928,17.9350,17.5824
7,9333,2025-04-21,put,6M,182,0.8500,20.5319,19.8634,structured_region_blend_callput_shrink,wing,False,3,19.8634,20.8219,20.5784
8,9372,2025-04-22,call,1M,30,0.9750,21.0011,20.9739,quadratic_smile_only,center,False,1,20.9739,20.2771,20.9259
9,9373,2025-04-22,put,1M,30,0.9750,20.6774,21.5864,quadratic_smile_only,center,False,2,21.5864,21.1058,21.7146


In [91]:
def apply_hybrid_center_structured_else_histgb(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_center_structured_else_histgb"
    out["hybrid_route"] = np.where(
        out["wing_center_bucket"] == "center",
        "structured_winner",
        "histgb_residual_full",
    )
    out["iv_pred"] = np.where(
        out["wing_center_bucket"] == "center",
        out["structured_winner__iv_pred"],
        out["histgb_residual_full__iv_pred"],
    )

    return out


def apply_hybrid_slice_rule_v1(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    # Default to the strongest optimistic nonlinear residual model.
    out["hybrid_model"] = "hybrid_slice_rule_v1"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    # Conservative overrides where Ridge looked stronger in diagnostics.
    ridge_mask = (
        out["hard_case"]
        | (out["structured_winner_source"] == "tv_maturity_only")
    )
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    # Protect slices where the structured winner already looked strongest.
    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


In [92]:
def score_prediction_frame(pred_df: pd.DataFrame) -> dict:
    return {
        "n": len(pred_df),
        "mae": (pred_df["target_iv"] - pred_df["iv_pred"]).abs().mean(),
        "rmse": rmse_from_series(pred_df["target_iv"], pred_df["iv_pred"]),
    }


HYBRID_BUILDERS = {
    "hybrid_center_structured_else_histgb": apply_hybrid_center_structured_else_histgb,
    "hybrid_slice_rule_v1": apply_hybrid_slice_rule_v1,
}

hybrid_rows = []
hybrid_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        panel_df = build_prediction_panel_for_fold_protocol(
            protocol_name,
            fold_id,
            table_cache=table_cache,
            prediction_cache=prediction_cache,
        )

        for hybrid_name, builder in HYBRID_BUILDERS.items():
            pred_df = builder(panel_df).copy()
            metrics = score_prediction_frame(pred_df)

            hybrid_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": hybrid_name,
                    "n_val_rows": len(pred_df),
                    "val_rmse": metrics["rmse"],
                    "val_mae": metrics["mae"],
                }
            )

            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            hybrid_prediction_cache[(protocol_name, fold_id, hybrid_name)] = pred_df

hybrid_results = pd.DataFrame(hybrid_rows).sort_values(["protocol", "fold", "val_rmse"]).reset_index(drop=True)

print("Hybrid fold-level results")
display(hybrid_results)


Hybrid fold-level results


,protocol,fold,model,n_val_rows,val_rmse,val_mae
0,primary_realistic,1,hybrid_center_structured_else_histgb,39,0.8932,0.6964
1,primary_realistic,1,hybrid_slice_rule_v1,39,0.9028,0.6654
2,primary_realistic,2,hybrid_center_structured_else_histgb,38,0.8616,0.7084
3,primary_realistic,2,hybrid_slice_rule_v1,38,0.9847,0.7641
4,primary_realistic,3,hybrid_slice_rule_v1,39,0.8137,0.6992
5,primary_realistic,3,hybrid_center_structured_else_histgb,39,0.8216,0.7102
6,primary_realistic,4,hybrid_slice_rule_v1,40,0.8447,0.6293
7,primary_realistic,4,hybrid_center_structured_else_histgb,40,0.8562,0.6352
8,stress_test,1,hybrid_slice_rule_v1,39,0.8647,0.7058
9,stress_test,1,hybrid_center_structured_else_histgb,39,1.0963,0.8656


In [93]:
frontline_summary_for_compare = (
    multi_fold_results[["protocol", "fold", "model", "val_rmse"]]
    .copy()
)

hybrid_compare_results = pd.concat(
    [
        frontline_summary_for_compare,
        hybrid_results[["protocol", "fold", "model", "val_rmse"]],
    ],
    ignore_index=True,
)

hybrid_protocol_summary = (
    hybrid_compare_results
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Hybrid protocol-level summary")
display(hybrid_protocol_summary)


Hybrid protocol-level summary


,protocol,model,mean_rmse,min_rmse,max_rmse
1,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932
0,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928
5,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292
2,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847
3,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556
4,primary_realistic,structured_winner,1.0638,0.7840,1.3592
8,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647
9,stress_test,ridge_direct_full,0.8463,0.7246,1.0449
7,stress_test,hybrid_center_structured_else_histgb,0.8949,0.8099,1.0963
11,stress_test,xgb_residual_full,0.9224,0.8464,1.0579


In [94]:
structured_ref = (
    hybrid_protocol_summary.loc[
        hybrid_protocol_summary["model"] == "structured_winner",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

hybrid_gain_table = (
    hybrid_protocol_summary
    .merge(structured_ref, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Hybrid gain/loss vs structured winner")
display(hybrid_gain_table)


Hybrid gain/loss vs structured winner


,protocol,model,mean_rmse,min_rmse,max_rmse,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932,1.0638,-0.2057
1,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928,1.0638,-0.1915
2,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292,1.0638,-0.1882
3,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847,1.0638,-0.1773
4,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,1.0638,-0.1393
5,primary_realistic,structured_winner,1.0638,0.7840,1.3592,1.0638,0.0000
6,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647,1.1847,-0.3933
7,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,1.1847,-0.3384
8,stress_test,hybrid_center_structured_else_histgb,0.8949,0.8099,1.0963,1.1847,-0.2898
9,stress_test,xgb_residual_full,0.9224,0.8464,1.0579,1.1847,-0.2623


In [95]:
hybrid_route_rows = []

for (protocol_name, fold_id, hybrid_name), pred_df in hybrid_prediction_cache.items():
    route_mix = (
        pred_df["hybrid_route"]
        .value_counts(dropna=False)
        .rename_axis("hybrid_route")
        .reset_index(name="count")
    )
    route_mix["protocol"] = protocol_name
    route_mix["fold"] = fold_id
    route_mix["model"] = hybrid_name
    hybrid_route_rows.append(route_mix)

hybrid_route_mix = pd.concat(hybrid_route_rows, ignore_index=True)

print("Hybrid route mix")
display(
    hybrid_route_mix
    .groupby(["protocol", "model", "hybrid_route"], as_index=False)["count"]
    .sum()
    .sort_values(["protocol", "model", "count"], ascending=[True, True, False])
)


Hybrid route mix


,protocol,model,hybrid_route,count
0,primary_realistic,hybrid_center_structured_else_histgb,histgb_residual_full,91
1,primary_realistic,hybrid_center_structured_else_histgb,structured_winner,65
4,primary_realistic,hybrid_slice_rule_v1,structured_winner,88
2,primary_realistic,hybrid_slice_rule_v1,histgb_residual_full,52
3,primary_realistic,hybrid_slice_rule_v1,ridge_direct_full,16
5,stress_test,hybrid_center_structured_else_histgb,histgb_residual_full,107
6,stress_test,hybrid_center_structured_else_histgb,structured_winner,49
9,stress_test,hybrid_slice_rule_v1,structured_winner,74
7,stress_test,hybrid_slice_rule_v1,histgb_residual_full,62
8,stress_test,hybrid_slice_rule_v1,ridge_direct_full,20


In [96]:
hybrid_fold_rank_view = (
    hybrid_compare_results
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Fold-level ranking view including hybrids")
display(hybrid_fold_rank_view)


Fold-level ranking view including hybrids


,protocol,fold,model,val_rmse
0,primary_realistic,1,histgb_residual_full,0.8914
1,primary_realistic,1,hybrid_center_structured_else_histgb,0.8932
2,primary_realistic,1,hybrid_slice_rule_v1,0.9028
3,primary_realistic,1,xgb_residual_full,0.9131
4,primary_realistic,1,ridge_direct_full,1.0556
5,primary_realistic,1,structured_winner,1.3592
6,primary_realistic,2,hybrid_center_structured_else_histgb,0.8616
7,primary_realistic,2,histgb_residual_full,0.8928
8,primary_realistic,2,xgb_residual_full,0.9292
9,primary_realistic,2,ridge_direct_full,0.9294


In [99]:
HYBRID_DECISION_MODELS = [
    "structured_winner",
    "ridge_direct_full",
    "histgb_residual_full",
    "hybrid_center_structured_else_histgb",
    "hybrid_slice_rule_v1",
]


def build_decision_diag_df(
    table_cache: dict,
    prediction_cache: dict,
    hybrid_prediction_cache: dict,
    model_names=None,
) -> pd.DataFrame:
    model_names = model_names or HYBRID_DECISION_MODELS
    rows = []

    for protocol_name in PROTOCOLS:
        for _, fold_row in fold_plan.iterrows():
            fold_id = int(fold_row["fold"])
            val_table = table_cache[(protocol_name, fold_id)]["val"].copy()

            base_cols = [
                "row_id",
                "target_date",
                "option_type",
                "maturity_label",
                "maturity_days",
                "moneyness",
                "target_iv",
                "wing_center_bucket",
                "hard_case",
                "structured_winner_source",
                "true_local_support_score",
                "same_maturity_visible_anchor_count",
                "opp_option_visible",
            ]

            base = val_table[base_cols].copy()
            base["hard_case_bucket"] = np.where(base["hard_case"], "hard_case", "non_hard_case")

            for model_name in model_names:
                cache_key = (protocol_name, fold_id, model_name)

                if cache_key in prediction_cache:
                    pred_df = prediction_cache[cache_key].copy()
                elif cache_key in hybrid_prediction_cache:
                    pred_df = hybrid_prediction_cache[cache_key].copy()
                else:
                    raise KeyError(f"Missing prediction cache entry for {cache_key}")

                merged = base.merge(
                    pred_df[["row_id", "iv_pred"]],
                    on="row_id",
                    how="left",
                )
                merged["protocol"] = protocol_name
                merged["fold"] = fold_id
                merged["model"] = model_name
                merged["abs_error"] = (merged["target_iv"] - merged["iv_pred"]).abs()
                merged["sq_error"] = (merged["target_iv"] - merged["iv_pred"]) ** 2
                rows.append(merged)

    return pd.concat(rows, ignore_index=True)


In [100]:
decision_diag_df = build_decision_diag_df(
    table_cache=table_cache,
    prediction_cache=prediction_cache,
    hybrid_prediction_cache=hybrid_prediction_cache,
    model_names=HYBRID_DECISION_MODELS,
)

decision_overall = slice_error_table(decision_diag_df, group_cols=[])

print("Decision-stage overall pooled ranking")
display(decision_overall)


Decision-stage overall pooled ranking


,protocol,model,n,mae,rmse
0,primary_realistic,histgb_residual_full,156,0.7039,0.8724
1,primary_realistic,hybrid_center_structured_else_histgb,156,0.6871,0.8585
2,primary_realistic,hybrid_slice_rule_v1,156,0.6886,0.8879
3,primary_realistic,ridge_direct_full,156,0.7109,0.9276
4,primary_realistic,structured_winner,156,0.7831,1.0834
5,stress_test,histgb_residual_full,156,0.7638,0.9475
6,stress_test,hybrid_center_structured_else_histgb,156,0.7052,0.9026
7,stress_test,hybrid_slice_rule_v1,156,0.6446,0.7935
8,stress_test,ridge_direct_full,156,0.6870,0.8542
9,stress_test,structured_winner,156,0.8722,1.1883


In [101]:
hybrid_only_df = decision_diag_df.loc[
    decision_diag_df["model"].isin(
        ["hybrid_center_structured_else_histgb", "hybrid_slice_rule_v1"]
    )
].copy()

hybrid_hard_case = slice_error_table(
    hybrid_only_df,
    group_cols=["hard_case_bucket"],
)

hybrid_wing_center = slice_error_table(
    hybrid_only_df,
    group_cols=["wing_center_bucket"],
)

hybrid_maturity = slice_error_table(
    hybrid_only_df,
    group_cols=["maturity_label"],
)

hybrid_source = slice_error_table(
    hybrid_only_df,
    group_cols=["structured_winner_source"],
)

print("Hybrid vs hybrid | hard-case")
display(hybrid_hard_case)

print("Hybrid vs hybrid | wing vs center")
display(hybrid_wing_center)

print("Hybrid vs hybrid | maturity")
display(hybrid_maturity)

print("Hybrid vs hybrid | source")
display(hybrid_source)


Hybrid vs hybrid | hard-case


,protocol,model,hard_case_bucket,n,mae,rmse
0,primary_realistic,hybrid_center_structured_else_histgb,hard_case,6,1.0209,1.3506
1,primary_realistic,hybrid_center_structured_else_histgb,non_hard_case,150,0.6737,0.8327
2,primary_realistic,hybrid_slice_rule_v1,hard_case,6,1.6564,2.0690
3,primary_realistic,hybrid_slice_rule_v1,non_hard_case,150,0.6499,0.8055
4,stress_test,hybrid_center_structured_else_histgb,hard_case,12,0.9574,1.2946
5,stress_test,hybrid_center_structured_else_histgb,non_hard_case,144,0.6842,0.8620
6,stress_test,hybrid_slice_rule_v1,hard_case,12,0.9553,1.1479
7,stress_test,hybrid_slice_rule_v1,non_hard_case,144,0.6187,0.7565


Hybrid vs hybrid | wing vs center


,protocol,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,hybrid_center_structured_else_histgb,center,65,0.5069,0.6537
1,primary_realistic,hybrid_center_structured_else_histgb,wing,91,0.8157,0.9788
2,primary_realistic,hybrid_slice_rule_v1,center,65,0.5069,0.6537
3,primary_realistic,hybrid_slice_rule_v1,wing,91,0.8184,1.0229
4,stress_test,hybrid_center_structured_else_histgb,center,49,0.5279,0.6479
5,stress_test,hybrid_center_structured_else_histgb,wing,107,0.7864,0.9978
6,stress_test,hybrid_slice_rule_v1,center,49,0.5279,0.6479
7,stress_test,hybrid_slice_rule_v1,wing,107,0.6981,0.8519


Hybrid vs hybrid | maturity


,protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,hybrid_center_structured_else_histgb,1M,40,0.7382,0.9369
1,primary_realistic,hybrid_center_structured_else_histgb,2M,39,0.7034,0.8931
2,primary_realistic,hybrid_center_structured_else_histgb,3M,39,0.6734,0.7758
3,primary_realistic,hybrid_center_structured_else_histgb,6M,38,0.6305,0.8155
4,primary_realistic,hybrid_slice_rule_v1,1M,40,0.6885,0.9580
5,primary_realistic,hybrid_slice_rule_v1,2M,39,0.7735,1.0045
6,primary_realistic,hybrid_slice_rule_v1,3M,39,0.6875,0.7935
7,primary_realistic,hybrid_slice_rule_v1,6M,38,0.6029,0.7675
8,stress_test,hybrid_center_structured_else_histgb,1M,40,0.5636,0.7525
9,stress_test,hybrid_center_structured_else_histgb,2M,39,0.8812,1.0768


Hybrid vs hybrid | source


,protocol,model,structured_winner_source,n,mae,rmse
0,primary_realistic,hybrid_center_structured_else_histgb,quadratic_smile_only,34,0.7563,0.8903
1,primary_realistic,hybrid_center_structured_else_histgb,same_date_linear_interp,18,0.8798,1.1616
2,primary_realistic,hybrid_center_structured_else_histgb,structured_region_blend_callput_shrink,89,0.6346,0.7908
3,primary_realistic,hybrid_center_structured_else_histgb,structured_region_blend_center,4,0.4193,0.4273
4,primary_realistic,hybrid_center_structured_else_histgb,structured_region_blend_wing,1,0.4038,0.4038
5,primary_realistic,hybrid_center_structured_else_histgb,tv_maturity_only,10,0.7071,0.8467
6,primary_realistic,hybrid_slice_rule_v1,quadratic_smile_only,34,0.6613,0.7944
7,primary_realistic,hybrid_slice_rule_v1,same_date_linear_interp,18,1.0917,1.4725
8,primary_realistic,hybrid_slice_rule_v1,structured_region_blend_callput_shrink,89,0.6346,0.7908
9,primary_realistic,hybrid_slice_rule_v1,structured_region_blend_center,4,0.4193,0.4273


In [102]:
def hybrid_winner_by_slice(slice_table: pd.DataFrame, slice_col: str) -> pd.DataFrame:
    return (
        slice_table
        .sort_values(["protocol", slice_col, "rmse"])
        .groupby(["protocol", slice_col], as_index=False)
        .first()[["protocol", slice_col, "model", "rmse", "n"]]
        .rename(columns={"model": "best_hybrid", "rmse": "best_rmse"})
    )


print("Best hybrid by hard-case bucket")
display(hybrid_winner_by_slice(hybrid_hard_case, "hard_case_bucket"))

print("Best hybrid by wing/center")
display(hybrid_winner_by_slice(hybrid_wing_center, "wing_center_bucket"))

print("Best hybrid by maturity")
display(hybrid_winner_by_slice(hybrid_maturity, "maturity_label"))


Best hybrid by hard-case bucket


,protocol,hard_case_bucket,best_hybrid,best_rmse,n
0,primary_realistic,hard_case,hybrid_center_structured_else_histgb,1.3506,6
1,primary_realistic,non_hard_case,hybrid_slice_rule_v1,0.8055,150
2,stress_test,hard_case,hybrid_slice_rule_v1,1.1479,12
3,stress_test,non_hard_case,hybrid_slice_rule_v1,0.7565,144


Best hybrid by wing/center


,protocol,wing_center_bucket,best_hybrid,best_rmse,n
0,primary_realistic,center,hybrid_center_structured_else_histgb,0.6537,65
1,primary_realistic,wing,hybrid_center_structured_else_histgb,0.9788,91
2,stress_test,center,hybrid_center_structured_else_histgb,0.6479,49
3,stress_test,wing,hybrid_slice_rule_v1,0.8519,107


Best hybrid by maturity


,protocol,maturity_label,best_hybrid,best_rmse,n
0,primary_realistic,1M,hybrid_center_structured_else_histgb,0.9369,40
1,primary_realistic,2M,hybrid_center_structured_else_histgb,0.8931,39
2,primary_realistic,3M,hybrid_center_structured_else_histgb,0.7758,39
3,primary_realistic,6M,hybrid_slice_rule_v1,0.7675,38
4,stress_test,1M,hybrid_slice_rule_v1,0.7145,40
5,stress_test,2M,hybrid_slice_rule_v1,0.9356,39
6,stress_test,3M,hybrid_slice_rule_v1,0.6487,39
7,stress_test,6M,hybrid_slice_rule_v1,0.8469,38


In [103]:
hybrid_protocol_only = hybrid_protocol_summary.loc[
    hybrid_protocol_summary["model"].isin(
        ["hybrid_center_structured_else_histgb", "hybrid_slice_rule_v1"]
    )
].copy()

hybrid_protocol_pivot = (
    hybrid_protocol_only
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

# Lower is better on both columns.
hybrid_protocol_pivot["avg_two_protocols"] = (
    hybrid_protocol_pivot["primary_realistic"] + hybrid_protocol_pivot["stress_test"]
) / 2.0

hybrid_protocol_pivot["max_two_protocols"] = hybrid_protocol_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Hybrid decision summary")
display(hybrid_protocol_pivot.sort_values(["max_two_protocols", "avg_two_protocols"]))


Hybrid decision summary


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
1,hybrid_slice_rule_v1,0.8865,0.7914,0.8389,0.8865
0,hybrid_center_structured_else_histgb,0.8581,0.8949,0.8765,0.8949


In [104]:
hybrid_fold_only = hybrid_results.copy()

hybrid_fold_wins = (
    hybrid_fold_only
    .sort_values(["protocol", "fold", "val_rmse"])
    .groupby(["protocol", "fold"], as_index=False)
    .first()[["protocol", "fold", "model", "val_rmse"]]
    .rename(columns={"model": "fold_winner", "val_rmse": "winning_rmse"})
)

print("Fold winners between the two hybrids")
display(hybrid_fold_wins)

print("Fold-win counts")
display(
    hybrid_fold_wins["fold_winner"]
    .value_counts()
    .rename_axis("fold_winner")
    .to_frame("count")
)


Fold winners between the two hybrids


,protocol,fold,fold_winner,winning_rmse
0,primary_realistic,1,hybrid_center_structured_else_histgb,0.8932
1,primary_realistic,2,hybrid_center_structured_else_histgb,0.8616
2,primary_realistic,3,hybrid_slice_rule_v1,0.8137
3,primary_realistic,4,hybrid_slice_rule_v1,0.8447
4,stress_test,1,hybrid_slice_rule_v1,0.8647
5,stress_test,2,hybrid_slice_rule_v1,0.7310
6,stress_test,3,hybrid_slice_rule_v1,0.7609
7,stress_test,4,hybrid_slice_rule_v1,0.8090


Fold-win counts


,count
fold_winner,
hybrid_slice_rule_v1,6
hybrid_center_structured_else_histgb,2


In [105]:
hybrid_head_to_head = (
    hybrid_results
    .pivot(index=["protocol", "fold"], columns="model", values="val_rmse")
    .reset_index()
)

hybrid_head_to_head["slice_rule_minus_center_histgb"] = (
    hybrid_head_to_head["hybrid_slice_rule_v1"]
    - hybrid_head_to_head["hybrid_center_structured_else_histgb"]
)

print("Head-to-head hybrid comparison by fold")
display(hybrid_head_to_head)

print("Head-to-head summary by protocol")
display(
    hybrid_head_to_head.groupby("protocol")["slice_rule_minus_center_histgb"]
    .agg(["mean", "min", "max"])
    .reset_index()
)


Head-to-head hybrid comparison by fold


model,protocol,fold,hybrid_center_structured_else_histgb,hybrid_slice_rule_v1,slice_rule_minus_center_histgb
0,primary_realistic,1,0.8932,0.9028,0.0097
1,primary_realistic,2,0.8616,0.9847,0.1232
2,primary_realistic,3,0.8216,0.8137,-0.0079
3,primary_realistic,4,0.8562,0.8447,-0.0115
4,stress_test,1,1.0963,0.8647,-0.2316
5,stress_test,2,0.8122,0.7310,-0.0812
6,stress_test,3,0.8612,0.7609,-0.1003
7,stress_test,4,0.8099,0.8090,-0.0008


Head-to-head summary by protocol


,protocol,mean,min,max
0,primary_realistic,0.0284,-0.0115,0.1232
1,stress_test,-0.1035,-0.2316,-0.0008


In [106]:
def apply_hybrid_slice_no_ridge(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep the structured protections, but remove Ridge entirely.
    This tests whether the Ridge override is the main source of the extra robustness.
    """
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_ridge"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_slice_no_hard_case_override(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep the tv_maturity_only Ridge override, but remove the hard_case Ridge override.
    """
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_hard_case_override"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    ridge_mask = (out["structured_winner_source"] == "tv_maturity_only")
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_slice_no_tv_override(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep the hard_case Ridge override, but remove the tv_maturity_only Ridge override.
    """
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_tv_override"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    ridge_mask = out["hard_case"]
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


def apply_hybrid_slice_no_quadratic_protection(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep the Ridge overrides, but remove the structured protection for quadratic_smile_only rows.
    """
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_quadratic_protection"
    out["hybrid_route"] = "histgb_residual_full"
    out["iv_pred"] = out["histgb_residual_full__iv_pred"]

    ridge_mask = (
        out["hard_case"]
        | (out["structured_winner_source"] == "tv_maturity_only")
    )
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (out["wing_center_bucket"] == "center")
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


In [107]:
ABLATION_BUILDERS = {
    "hybrid_center_structured_else_histgb": apply_hybrid_center_structured_else_histgb,
    "hybrid_slice_rule_v1": apply_hybrid_slice_rule_v1,
    "hybrid_slice_no_ridge": apply_hybrid_slice_no_ridge,
    "hybrid_slice_no_hard_case_override": apply_hybrid_slice_no_hard_case_override,
    "hybrid_slice_no_tv_override": apply_hybrid_slice_no_tv_override,
    "hybrid_slice_no_quadratic_protection": apply_hybrid_slice_no_quadratic_protection,
}

ablation_rows = []
ablation_prediction_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        panel_df = build_prediction_panel_for_fold_protocol(
            protocol_name,
            fold_id,
            table_cache=table_cache,
            prediction_cache=prediction_cache,
        )

        for model_name, builder in ABLATION_BUILDERS.items():
            pred_df = builder(panel_df).copy()
            metrics = score_prediction_frame(pred_df)

            ablation_rows.append(
                {
                    "protocol": protocol_name,
                    "fold": fold_id,
                    "model": model_name,
                    "n_val_rows": len(pred_df),
                    "val_rmse": metrics["rmse"],
                    "val_mae": metrics["mae"],
                }
            )

            pred_df["protocol"] = protocol_name
            pred_df["fold"] = fold_id
            ablation_prediction_cache[(protocol_name, fold_id, model_name)] = pred_df

ablation_results = (
    pd.DataFrame(ablation_rows)
    .sort_values(["protocol", "fold", "val_rmse"])
    .reset_index(drop=True)
)

print("Ablation fold-level results")
display(ablation_results)


Ablation fold-level results


,protocol,fold,model,n_val_rows,val_rmse,val_mae
0,primary_realistic,1,hybrid_slice_no_hard_case_override,39,0.8084,0.6257
1,primary_realistic,1,hybrid_slice_no_ridge,39,0.8488,0.6607
2,primary_realistic,1,hybrid_center_structured_else_histgb,39,0.8932,0.6964
3,primary_realistic,1,hybrid_slice_rule_v1,39,0.9028,0.6654
4,primary_realistic,1,hybrid_slice_no_tv_override,39,0.9391,0.7005
5,primary_realistic,1,hybrid_slice_no_quadratic_protection,39,0.9447,0.7011
6,primary_realistic,2,hybrid_slice_no_ridge,38,0.8502,0.6907
7,primary_realistic,2,hybrid_center_structured_else_histgb,38,0.8616,0.7084
8,primary_realistic,2,hybrid_slice_no_hard_case_override,38,0.8620,0.6988
9,primary_realistic,2,hybrid_slice_no_tv_override,38,0.9744,0.7560


In [109]:
ablation_protocol_summary = (
    ablation_results
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Ablation protocol-level summary")
display(ablation_protocol_summary)


Ablation protocol-level summary


,protocol,model,mean_rmse,min_rmse,max_rmse
1,primary_realistic,hybrid_slice_no_hard_case_override,0.8328,0.8084,0.8620
3,primary_realistic,hybrid_slice_no_ridge,0.8375,0.8078,0.8502
0,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932
5,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847
4,primary_realistic,hybrid_slice_no_tv_override,0.8906,0.8078,0.9744
2,primary_realistic,hybrid_slice_no_quadratic_protection,0.9060,0.8273,0.9946
11,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647
7,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624
10,stress_test,hybrid_slice_no_tv_override,0.8389,0.7571,1.0033
8,stress_test,hybrid_slice_no_quadratic_protection,0.8460,0.7880,0.9937


In [110]:
ablation_focus = ablation_protocol_summary.loc[
    ablation_protocol_summary["model"].isin(ABLATION_BUILDERS.keys())
].copy()

ablation_pivot = (
    ablation_focus
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

ablation_pivot["avg_two_protocols"] = (
    ablation_pivot["primary_realistic"] + ablation_pivot["stress_test"]
) / 2.0

ablation_pivot["max_two_protocols"] = ablation_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Ablation decision pivot")
display(ablation_pivot.sort_values(["max_two_protocols", "avg_two_protocols"]))


Ablation decision pivot


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
1,hybrid_slice_no_hard_case_override,0.8328,0.8033,0.8181,0.8328
3,hybrid_slice_no_ridge,0.8375,0.8506,0.8441,0.8506
5,hybrid_slice_rule_v1,0.8865,0.7914,0.8389,0.8865
4,hybrid_slice_no_tv_override,0.8906,0.8389,0.8647,0.8906
0,hybrid_center_structured_else_histgb,0.8581,0.8949,0.8765,0.8949
2,hybrid_slice_no_quadratic_protection,0.9060,0.8460,0.8760,0.9060


In [111]:
slice_rule_ref = (
    ablation_protocol_summary.loc[
        ablation_protocol_summary["model"] == "hybrid_slice_rule_v1",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "slice_rule_mean_rmse"})
)

ablation_delta_vs_slice_rule = (
    ablation_protocol_summary
    .merge(slice_rule_ref, on="protocol", how="left")
    .assign(delta_vs_slice_rule=lambda df: df["mean_rmse"] - df["slice_rule_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Ablation delta vs hybrid_slice_rule_v1")
display(ablation_delta_vs_slice_rule)


Ablation delta vs hybrid_slice_rule_v1


,protocol,model,mean_rmse,min_rmse,max_rmse,slice_rule_mean_rmse,delta_vs_slice_rule
0,primary_realistic,hybrid_slice_no_hard_case_override,0.8328,0.8084,0.8620,0.8865,-0.0536
1,primary_realistic,hybrid_slice_no_ridge,0.8375,0.8078,0.8502,0.8865,-0.0489
2,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932,0.8865,-0.0284
3,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847,0.8865,0.0000
4,primary_realistic,hybrid_slice_no_tv_override,0.8906,0.8078,0.9744,0.8865,0.0041
5,primary_realistic,hybrid_slice_no_quadratic_protection,0.9060,0.8273,0.9946,0.8865,0.0195
6,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647,0.7914,0.0000
7,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624,0.7914,0.0119
8,stress_test,hybrid_slice_no_tv_override,0.8389,0.7571,1.0033,0.7914,0.0475
9,stress_test,hybrid_slice_no_quadratic_protection,0.8460,0.7880,0.9937,0.7914,0.0546


In [112]:
ablation_route_rows = []

for (protocol_name, fold_id, model_name), pred_df in ablation_prediction_cache.items():
    route_mix = (
        pred_df["hybrid_route"]
        .value_counts(dropna=False)
        .rename_axis("hybrid_route")
        .reset_index(name="count")
    )
    route_mix["protocol"] = protocol_name
    route_mix["fold"] = fold_id
    route_mix["model"] = model_name
    ablation_route_rows.append(route_mix)

ablation_route_mix = pd.concat(ablation_route_rows, ignore_index=True)

print("Ablation route mix")
display(
    ablation_route_mix
    .groupby(["protocol", "model", "hybrid_route"], as_index=False)["count"]
    .sum()
    .sort_values(["protocol", "model", "count"], ascending=[True, True, False])
)


Ablation route mix


,protocol,model,hybrid_route,count
0,primary_realistic,hybrid_center_structured_else_histgb,histgb_residual_full,91
1,primary_realistic,hybrid_center_structured_else_histgb,structured_winner,65
4,primary_realistic,hybrid_slice_no_hard_case_override,structured_winner,88
2,primary_realistic,hybrid_slice_no_hard_case_override,histgb_residual_full,58
3,primary_realistic,hybrid_slice_no_hard_case_override,ridge_direct_full,10
5,primary_realistic,hybrid_slice_no_quadratic_protection,histgb_residual_full,75
7,primary_realistic,hybrid_slice_no_quadratic_protection,structured_winner,65
6,primary_realistic,hybrid_slice_no_quadratic_protection,ridge_direct_full,16
9,primary_realistic,hybrid_slice_no_ridge,structured_winner,88
8,primary_realistic,hybrid_slice_no_ridge,histgb_residual_full,68


In [113]:
stress_ablation_summary = (
    ablation_protocol_summary.loc[ablation_protocol_summary["protocol"] == "stress_test"]
    .sort_values("mean_rmse")
    .reset_index(drop=True)
)

print("Stress-test-only ablation summary")
display(stress_ablation_summary)


Stress-test-only ablation summary


,protocol,model,mean_rmse,min_rmse,max_rmse
0,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647
1,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624
2,stress_test,hybrid_slice_no_tv_override,0.8389,0.7571,1.0033
3,stress_test,hybrid_slice_no_quadratic_protection,0.8460,0.7880,0.9937
4,stress_test,hybrid_slice_no_ridge,0.8506,0.7571,1.0013
5,stress_test,hybrid_center_structured_else_histgb,0.8949,0.8099,1.0963


In [114]:
FINAL_CANDIDATE_MODELS = [
    "structured_winner",
    "ridge_direct_full",
    "histgb_residual_full",
    "xgb_residual_full",
    "hybrid_center_structured_else_histgb",
    "hybrid_slice_rule_v1",
    "hybrid_slice_no_hard_case_override",
]


def collect_model_results_for_final_leaderboard(
    multi_fold_results: pd.DataFrame,
    hybrid_results: pd.DataFrame,
    ablation_results: pd.DataFrame,
    model_names=None,
) -> pd.DataFrame:
    model_names = model_names or FINAL_CANDIDATE_MODELS

    all_results = pd.concat(
        [
            multi_fold_results[["protocol", "fold", "model", "val_rmse"]],
            hybrid_results[["protocol", "fold", "model", "val_rmse"]],
            ablation_results[["protocol", "fold", "model", "val_rmse"]],
        ],
        ignore_index=True,
    )

    all_results = all_results.loc[all_results["model"].isin(model_names)].copy()
    all_results = all_results.drop_duplicates(subset=["protocol", "fold", "model"], keep="last")

    return all_results.sort_values(["protocol", "fold", "val_rmse"]).reset_index(drop=True)


final_leaderboard_rows = collect_model_results_for_final_leaderboard(
    multi_fold_results=multi_fold_results,
    hybrid_results=hybrid_results,
    ablation_results=ablation_results,
)

print("Final candidate fold-level results")
display(final_leaderboard_rows)


Final candidate fold-level results


,protocol,fold,model,val_rmse
0,primary_realistic,1,hybrid_slice_no_hard_case_override,0.8084
1,primary_realistic,1,histgb_residual_full,0.8914
2,primary_realistic,1,hybrid_center_structured_else_histgb,0.8932
3,primary_realistic,1,hybrid_slice_rule_v1,0.9028
4,primary_realistic,1,xgb_residual_full,0.9131
5,primary_realistic,1,ridge_direct_full,1.0556
6,primary_realistic,1,structured_winner,1.3592
7,primary_realistic,2,hybrid_center_structured_else_histgb,0.8616
8,primary_realistic,2,hybrid_slice_no_hard_case_override,0.8620
9,primary_realistic,2,histgb_residual_full,0.8928


In [115]:
final_protocol_summary = (
    final_leaderboard_rows
    .groupby(["protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_rmse"])
)

print("Final candidate protocol-level summary")
display(final_protocol_summary)


Final candidate protocol-level summary


,protocol,model,mean_rmse,min_rmse,max_rmse
2,primary_realistic,hybrid_slice_no_hard_case_override,0.8328,0.8084,0.8620
1,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932
0,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928
6,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292
3,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847
4,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556
5,primary_realistic,structured_winner,1.0638,0.7840,1.3592
10,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647
9,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624
11,stress_test,ridge_direct_full,0.8463,0.7246,1.0449


In [116]:
final_protocol_pivot = (
    final_protocol_summary
    .pivot(index="model", columns="protocol", values="mean_rmse")
    .reset_index()
)

final_protocol_pivot["avg_two_protocols"] = (
    final_protocol_pivot["primary_realistic"] + final_protocol_pivot["stress_test"]
) / 2.0

final_protocol_pivot["max_two_protocols"] = final_protocol_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("Final candidate decision pivot")
display(final_protocol_pivot.sort_values(["max_two_protocols", "avg_two_protocols"]))


Final candidate decision pivot


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
2,hybrid_slice_no_hard_case_override,0.8328,0.8033,0.8181,0.8328
3,hybrid_slice_rule_v1,0.8865,0.7914,0.8389,0.8865
1,hybrid_center_structured_else_histgb,0.8581,0.8949,0.8765,0.8949
6,xgb_residual_full,0.8756,0.9224,0.8990,0.9224
4,ridge_direct_full,0.9245,0.8463,0.8854,0.9245
0,histgb_residual_full,0.8724,0.9415,0.9069,0.9415
5,structured_winner,1.0638,1.1847,1.1243,1.1847


In [117]:
structured_ref = (
    final_protocol_summary.loc[
        final_protocol_summary["model"] == "structured_winner",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

final_gain_table = (
    final_protocol_summary
    .merge(structured_ref, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Final candidate gain/loss vs structured winner")
display(final_gain_table)


Final candidate gain/loss vs structured winner


,protocol,model,mean_rmse,min_rmse,max_rmse,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,hybrid_slice_no_hard_case_override,0.8328,0.8084,0.8620,1.0638,-0.2310
1,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932,1.0638,-0.2057
2,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928,1.0638,-0.1915
3,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292,1.0638,-0.1882
4,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847,1.0638,-0.1773
5,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,1.0638,-0.1393
6,primary_realistic,structured_winner,1.0638,0.7840,1.3592,1.0638,0.0000
7,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647,1.1847,-0.3933
8,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624,1.1847,-0.3814
9,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,1.1847,-0.3384


In [118]:
structured_ref = (
    final_protocol_summary.loc[
        final_protocol_summary["model"] == "structured_winner",
        ["protocol", "mean_rmse"],
    ]
    .rename(columns={"mean_rmse": "structured_winner_mean_rmse"})
)

final_gain_table = (
    final_protocol_summary
    .merge(structured_ref, on="protocol", how="left")
    .assign(delta_vs_structured=lambda df: df["mean_rmse"] - df["structured_winner_mean_rmse"])
    .sort_values(["protocol", "mean_rmse"])
)

print("Final candidate gain/loss vs structured winner")
display(final_gain_table)


Final candidate gain/loss vs structured winner


,protocol,model,mean_rmse,min_rmse,max_rmse,structured_winner_mean_rmse,delta_vs_structured
0,primary_realistic,hybrid_slice_no_hard_case_override,0.8328,0.8084,0.8620,1.0638,-0.2310
1,primary_realistic,hybrid_center_structured_else_histgb,0.8581,0.8216,0.8932,1.0638,-0.2057
2,primary_realistic,histgb_residual_full,0.8724,0.8192,0.8928,1.0638,-0.1915
3,primary_realistic,xgb_residual_full,0.8756,0.8113,0.9292,1.0638,-0.1882
4,primary_realistic,hybrid_slice_rule_v1,0.8865,0.8137,0.9847,1.0638,-0.1773
5,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556,1.0638,-0.1393
6,primary_realistic,structured_winner,1.0638,0.7840,1.3592,1.0638,0.0000
7,stress_test,hybrid_slice_rule_v1,0.7914,0.7310,0.8647,1.1847,-0.3933
8,stress_test,hybrid_slice_no_hard_case_override,0.8033,0.7310,0.8624,1.1847,-0.3814
9,stress_test,ridge_direct_full,0.8463,0.7246,1.0449,1.1847,-0.3384


In [119]:
final_fold_winners = (
    final_leaderboard_rows
    .sort_values(["protocol", "fold", "val_rmse"])
    .groupby(["protocol", "fold"], as_index=False)
    .first()[["protocol", "fold", "model", "val_rmse"]]
    .rename(columns={"model": "fold_winner", "val_rmse": "winning_rmse"})
)

print("Fold winners across final candidates")
display(final_fold_winners)

print("Fold-win counts across final candidates")
display(
    final_fold_winners["fold_winner"]
    .value_counts()
    .rename_axis("fold_winner")
    .to_frame("count")
)


Fold winners across final candidates


,protocol,fold,fold_winner,winning_rmse
0,primary_realistic,1,hybrid_slice_no_hard_case_override,0.8084
1,primary_realistic,2,hybrid_center_structured_else_histgb,0.8616
2,primary_realistic,3,structured_winner,0.7840
3,primary_realistic,4,histgb_residual_full,0.8192
4,stress_test,1,hybrid_slice_no_hard_case_override,0.8624
5,stress_test,2,hybrid_slice_rule_v1,0.7310
6,stress_test,3,hybrid_slice_rule_v1,0.7609
7,stress_test,4,ridge_direct_full,0.7246


Fold-win counts across final candidates


,count
fold_winner,
hybrid_slice_no_hard_case_override,2
hybrid_slice_rule_v1,2
hybrid_center_structured_else_histgb,1
structured_winner,1
histgb_residual_full,1
ridge_direct_full,1


In [120]:
final_fold_winners = (
    final_leaderboard_rows
    .sort_values(["protocol", "fold", "val_rmse"])
    .groupby(["protocol", "fold"], as_index=False)
    .first()[["protocol", "fold", "model", "val_rmse"]]
    .rename(columns={"model": "fold_winner", "val_rmse": "winning_rmse"})
)

print("Fold winners across final candidates")
display(final_fold_winners)

print("Fold-win counts across final candidates")
display(
    final_fold_winners["fold_winner"]
    .value_counts()
    .rename_axis("fold_winner")
    .to_frame("count")
)


Fold winners across final candidates


,protocol,fold,fold_winner,winning_rmse
0,primary_realistic,1,hybrid_slice_no_hard_case_override,0.8084
1,primary_realistic,2,hybrid_center_structured_else_histgb,0.8616
2,primary_realistic,3,structured_winner,0.7840
3,primary_realistic,4,histgb_residual_full,0.8192
4,stress_test,1,hybrid_slice_no_hard_case_override,0.8624
5,stress_test,2,hybrid_slice_rule_v1,0.7310
6,stress_test,3,hybrid_slice_rule_v1,0.7609
7,stress_test,4,ridge_direct_full,0.7246


Fold-win counts across final candidates


,count
fold_winner,
hybrid_slice_no_hard_case_override,2
hybrid_slice_rule_v1,2
hybrid_center_structured_else_histgb,1
structured_winner,1
histgb_residual_full,1
ridge_direct_full,1


In [121]:
final_summary_table = final_protocol_pivot.sort_values(
    ["max_two_protocols", "avg_two_protocols"]
).reset_index(drop=True)

print("Final Phase 5 candidate summary")
display(final_summary_table)


Final Phase 5 candidate summary


protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,hybrid_slice_no_hard_case_override,0.8328,0.8033,0.8181,0.8328
1,hybrid_slice_rule_v1,0.8865,0.7914,0.8389,0.8865
2,hybrid_center_structured_else_histgb,0.8581,0.8949,0.8765,0.8949
3,xgb_residual_full,0.8756,0.9224,0.8990,0.9224
4,ridge_direct_full,0.9245,0.8463,0.8854,0.9245
5,histgb_residual_full,0.8724,0.9415,0.9069,0.9415
6,structured_winner,1.0638,1.1847,1.1243,1.1847


In [122]:
def apply_phase5_current_leader(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Current Phase 5 leader:
    - default to HistGB residual
    - protect center and quadratic_smile_only with structured_winner
    - use Ridge direct only for tv_maturity_only rows
    """
    return apply_hybrid_slice_no_hard_case_override(panel_df)


CURRENT_PHASE5_LEADER_NAME = "hybrid_slice_no_hard_case_override"
CURRENT_PHASE5_LEADER_LABEL = "phase5_current_leader"


In [123]:
leader_panel_example = build_prediction_panel_for_fold_protocol(
    "primary_realistic",
    1,
    table_cache=table_cache,
    prediction_cache=prediction_cache,
)

leader_example_preds = apply_phase5_current_leader(leader_panel_example).copy()

print("Current Phase 5 leader | route mix example")
display(
    leader_example_preds["hybrid_route"]
    .value_counts(dropna=False)
    .rename_axis("hybrid_route")
    .to_frame("count")
)


Current Phase 5 leader | route mix example


,count
hybrid_route,
structured_winner,19
histgb_residual_full,17
ridge_direct_full,3


In [124]:
model_catalog = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "family": "structured",
            "target_mode": "direct_iv",
            "model_type": "handcrafted",
            "routing_style": "single_model",
            "core_logic": "structured_region_blend_callput_shrink",
            "uses_ml": False,
            "uses_residual_learning": False,
            "uses_routing": False,
            "status": "benchmark",
            "notes": "Locked Phase 4 winner.",
        },
        {
            "model": "ridge_direct_full",
            "family": "linear_ml",
            "target_mode": "direct_iv",
            "model_type": "Ridge",
            "routing_style": "single_model",
            "core_logic": "full_feature_linear_direct_iv",
            "uses_ml": True,
            "uses_residual_learning": False,
            "uses_routing": False,
            "status": "reference",
            "notes": "Best standalone conservative/lightweight model.",
        },
        {
            "model": "histgb_residual_full",
            "family": "tree_ml",
            "target_mode": "residual",
            "model_type": "HistGradientBoostingRegressor",
            "routing_style": "single_model",
            "core_logic": "full_feature_nonlinear_residual",
            "uses_ml": True,
            "uses_residual_learning": True,
            "uses_routing": False,
            "status": "reference",
            "notes": "Best standalone nonlinear residual model on optimistic protocol.",
        },
        {
            "model": "xgb_residual_full",
            "family": "tree_ml",
            "target_mode": "residual",
            "model_type": "XGBoost",
            "routing_style": "single_model",
            "core_logic": "full_feature_xgb_residual",
            "uses_ml": True,
            "uses_residual_learning": True,
            "uses_routing": False,
            "status": "reference",
            "notes": "Strong residual model, but not final leader.",
        },
        {
            "model": "hybrid_center_structured_else_histgb",
            "family": "hybrid",
            "target_mode": "mixed",
            "model_type": "rule_based_hybrid",
            "routing_style": "center_vs_wing",
            "core_logic": "structured in center, HistGB residual elsewhere",
            "uses_ml": True,
            "uses_residual_learning": True,
            "uses_routing": True,
            "status": "runner_up",
            "notes": "Best optimistic hybrid.",
        },
        {
            "model": "hybrid_slice_rule_v1",
            "family": "hybrid",
            "target_mode": "mixed",
            "model_type": "rule_based_hybrid",
            "routing_style": "slice_gated",
            "core_logic": "HistGB residual default; structured on center/quadratic; Ridge on hard_case/tv_maturity_only",
            "uses_ml": True,
            "uses_residual_learning": True,
            "uses_routing": True,
            "status": "prior_leader",
            "notes": "Earlier robust leader before ablation simplification.",
        },
        {
            "model": "hybrid_slice_no_hard_case_override",
            "family": "hybrid",
            "target_mode": "mixed",
            "model_type": "rule_based_hybrid",
            "routing_style": "slice_gated",
            "core_logic": "HistGB residual default; structured on center/quadratic; Ridge only on tv_maturity_only",
            "uses_ml": True,
            "uses_residual_learning": True,
            "uses_routing": True,
            "status": "current_leader",
            "notes": "Current best balanced Phase 5 model after ablation.",
        },
    ]
)

print("Phase 5 model catalog")
display(model_catalog)


Phase 5 model catalog


,model,family,target_mode,model_type,routing_style,core_logic,uses_ml,uses_residual_learning,uses_routing,status,notes
0,structured_winner,structured,direct_iv,handcrafted,single_model,structured_region_blend_callput_shrink,False,False,False,benchmark,Locked Phase 4 winner.
1,ridge_direct_full,linear_ml,direct_iv,Ridge,single_model,full_feature_linear_direct_iv,True,False,False,reference,Best standalone conservative/lightweight model.
2,histgb_residual_full,tree_ml,residual,HistGradientBoostingRegressor,single_model,full_feature_nonlinear_residual,True,True,False,reference,Best standalone nonlinear residual model on op...
3,xgb_residual_full,tree_ml,residual,XGBoost,single_model,full_feature_xgb_residual,True,True,False,reference,"Strong residual model, but not final leader."
4,hybrid_center_structured_else_histgb,hybrid,mixed,rule_based_hybrid,center_vs_wing,"structured in center, HistGB residual elsewhere",True,True,True,runner_up,Best optimistic hybrid.
5,hybrid_slice_rule_v1,hybrid,mixed,rule_based_hybrid,slice_gated,HistGB residual default; structured on center/...,True,True,True,prior_leader,Earlier robust leader before ablation simplifi...
6,hybrid_slice_no_hard_case_override,hybrid,mixed,rule_based_hybrid,slice_gated,HistGB residual default; structured on center/...,True,True,True,current_leader,Current best balanced Phase 5 model after abla...


In [125]:
final_performance_registry = (
    model_catalog
    .merge(final_protocol_pivot, on="model", how="left")
    .merge(
        final_gain_table[["protocol", "model", "delta_vs_structured"]],
        on="model",
        how="left",
    )
)

# The merge above duplicates rows by protocol for delta_vs_structured.
# Re-aggregate into wide columns for cleaner reading.
delta_wide = (
    final_gain_table[["protocol", "model", "delta_vs_structured"]]
    .pivot(index="model", columns="protocol", values="delta_vs_structured")
    .reset_index()
    .rename(
        columns={
            "primary_realistic": "delta_vs_structured_primary_realistic",
            "stress_test": "delta_vs_structured_stress_test",
        }
    )
)

final_performance_registry = (
    model_catalog
    .merge(final_protocol_pivot, on="model", how="left")
    .merge(delta_wide, on="model", how="left")
    .sort_values(["max_two_protocols", "avg_two_protocols"])
    .reset_index(drop=True)
)

print("Phase 5 final performance registry")
display(final_performance_registry)


Phase 5 final performance registry


,model,family,target_mode,model_type,routing_style,core_logic,uses_ml,uses_residual_learning,uses_routing,status,notes,primary_realistic,stress_test,avg_two_protocols,max_two_protocols,delta_vs_structured_primary_realistic,delta_vs_structured_stress_test
0,hybrid_slice_no_hard_case_override,hybrid,mixed,rule_based_hybrid,slice_gated,HistGB residual default; structured on center/...,True,True,True,current_leader,Current best balanced Phase 5 model after abla...,0.8328,0.8033,0.8181,0.8328,-0.2310,-0.3814
1,hybrid_slice_rule_v1,hybrid,mixed,rule_based_hybrid,slice_gated,HistGB residual default; structured on center/...,True,True,True,prior_leader,Earlier robust leader before ablation simplifi...,0.8865,0.7914,0.8389,0.8865,-0.1773,-0.3933
2,hybrid_center_structured_else_histgb,hybrid,mixed,rule_based_hybrid,center_vs_wing,"structured in center, HistGB residual elsewhere",True,True,True,runner_up,Best optimistic hybrid.,0.8581,0.8949,0.8765,0.8949,-0.2057,-0.2898
3,xgb_residual_full,tree_ml,residual,XGBoost,single_model,full_feature_xgb_residual,True,True,False,reference,"Strong residual model, but not final leader.",0.8756,0.9224,0.8990,0.9224,-0.1882,-0.2623
4,ridge_direct_full,linear_ml,direct_iv,Ridge,single_model,full_feature_linear_direct_iv,True,False,False,reference,Best standalone conservative/lightweight model.,0.9245,0.8463,0.8854,0.9245,-0.1393,-0.3384
5,histgb_residual_full,tree_ml,residual,HistGradientBoostingRegressor,single_model,full_feature_nonlinear_residual,True,True,False,reference,Best standalone nonlinear residual model on op...,0.8724,0.9415,0.9069,0.9415,-0.1915,-0.2432
6,structured_winner,structured,direct_iv,handcrafted,single_model,structured_region_blend_callput_shrink,False,False,False,benchmark,Locked Phase 4 winner.,1.0638,1.1847,1.1243,1.1847,0.0000,0.0000


In [126]:
leaderboard_with_details = final_performance_registry[
    [
        "model",
        "status",
        "family",
        "model_type",
        "target_mode",
        "routing_style",
        "primary_realistic",
        "stress_test",
        "avg_two_protocols",
        "max_two_protocols",
        "delta_vs_structured_primary_realistic",
        "delta_vs_structured_stress_test",
        "notes",
    ]
].copy()

print("Phase 5 leaderboard with details")
display(leaderboard_with_details)


Phase 5 leaderboard with details


,model,status,family,model_type,target_mode,routing_style,primary_realistic,stress_test,avg_two_protocols,max_two_protocols,delta_vs_structured_primary_realistic,delta_vs_structured_stress_test,notes
0,hybrid_slice_no_hard_case_override,current_leader,hybrid,rule_based_hybrid,mixed,slice_gated,0.8328,0.8033,0.8181,0.8328,-0.2310,-0.3814,Current best balanced Phase 5 model after abla...
1,hybrid_slice_rule_v1,prior_leader,hybrid,rule_based_hybrid,mixed,slice_gated,0.8865,0.7914,0.8389,0.8865,-0.1773,-0.3933,Earlier robust leader before ablation simplifi...
2,hybrid_center_structured_else_histgb,runner_up,hybrid,rule_based_hybrid,mixed,center_vs_wing,0.8581,0.8949,0.8765,0.8949,-0.2057,-0.2898,Best optimistic hybrid.
3,xgb_residual_full,reference,tree_ml,XGBoost,residual,single_model,0.8756,0.9224,0.8990,0.9224,-0.1882,-0.2623,"Strong residual model, but not final leader."
4,ridge_direct_full,reference,linear_ml,Ridge,direct_iv,single_model,0.9245,0.8463,0.8854,0.9245,-0.1393,-0.3384,Best standalone conservative/lightweight model.
5,histgb_residual_full,reference,tree_ml,HistGradientBoostingRegressor,residual,single_model,0.8724,0.9415,0.9069,0.9415,-0.1915,-0.2432,Best standalone nonlinear residual model on op...
6,structured_winner,benchmark,structured,handcrafted,direct_iv,single_model,1.0638,1.1847,1.1243,1.1847,0.0000,0.0000,Locked Phase 4 winner.


In [127]:
hybrid_route_catalog = pd.DataFrame(
    [
        {
            "model": "hybrid_center_structured_else_histgb",
            "default_route": "histgb_residual_full",
            "structured_routes": "center -> structured_winner",
            "ridge_routes": "none",
        },
        {
            "model": "hybrid_slice_rule_v1",
            "default_route": "histgb_residual_full",
            "structured_routes": "center -> structured_winner; quadratic_smile_only -> structured_winner",
            "ridge_routes": "hard_case -> ridge_direct_full; tv_maturity_only -> ridge_direct_full",
        },
        {
            "model": "hybrid_slice_no_hard_case_override",
            "default_route": "histgb_residual_full",
            "structured_routes": "center -> structured_winner; quadratic_smile_only -> structured_winner",
            "ridge_routes": "tv_maturity_only -> ridge_direct_full",
        },
    ]
)

print("Hybrid route catalog")
display(hybrid_route_catalog)


Hybrid route catalog


,model,default_route,structured_routes,ridge_routes
0,hybrid_center_structured_else_histgb,histgb_residual_full,center -> structured_winner,none
1,hybrid_slice_rule_v1,histgb_residual_full,center -> structured_winner; quadratic_smile_o...,hard_case -> ridge_direct_full; tv_maturity_on...
2,hybrid_slice_no_hard_case_override,histgb_residual_full,center -> structured_winner; quadratic_smile_o...,tv_maturity_only -> ridge_direct_full


In [128]:
phase5_current_leader_snapshot = leaderboard_with_details.loc[
    leaderboard_with_details["model"] == CURRENT_PHASE5_LEADER_NAME
].copy()

print("Current Phase 5 leader snapshot")
display(phase5_current_leader_snapshot)


Current Phase 5 leader snapshot


,model,status,family,model_type,target_mode,routing_style,primary_realistic,stress_test,avg_two_protocols,max_two_protocols,delta_vs_structured_primary_realistic,delta_vs_structured_stress_test,notes
0,hybrid_slice_no_hard_case_override,current_leader,hybrid,rule_based_hybrid,mixed,slice_gated,0.8328,0.8033,0.8181,0.8328,-0.2310,-0.3814,Current best balanced Phase 5 model after abla...


## Phase 5.0 ML feature pipeline and first tabular-model results: final summary

This notebook is the bridge from the structured modeling work in Phase 4 to supervised tabular ML.

The goal here was **not** to run a broad final model sweep.
The goal was to answer a more careful question:

**Can we build a leakage-safe supervised feature pipeline on top of the Phase 4 structured winner, and does tabular ML add meaningful value beyond that structured benchmark?**

The locked Phase 4 benchmark carried into this notebook is:

- `structured_region_blend_callput_shrink`
- with `shrink_alpha = 0.25`

This model is referred to throughout Phase 5 as the **structured winner**.

---

## 1. What this notebook actually accomplished

This notebook had two jobs:

1. build the leakage-safe supervised learning pipeline
2. run the first controlled ML comparisons on top of that pipeline

So the output of this notebook is not just a score table.
It is a complete modeling handoff:

- leakage-safe feature generation
- pseudo-masked train rows
- matched validation rows
- structured-model features
- regime/date-level proxy features
- first direct-vs-residual ML comparisons
- first nonlinear model comparisons
- first hybrid routing experiments
- and a clear current Phase 5 leader

---

## 2. Leakage-safe supervised design

The most important Phase 5 requirement was avoiding leakage.

### Training example construction
Pseudo-masked training examples were created from:

- **originally observed train rows only**

For each target date:
- only **earlier dates** were used as historical training context
- same-date masking preserved visible anchors
- scored rows were the pseudo-hidden originally observed rows

So each supervised row represents:

- what would have been visible at prediction time
- plus the true hidden IV target we want ML to learn

### Structured-feature recomputation rule
A crucial rule in this notebook is:

**Structured predictions used as features for a target row are always recomputed under that row’s masked information set.**

That means:
- no structured feature can use the target row’s own hidden truth
- same-date features are computed only from visible same-date anchors
- historical features use only earlier dates

This is the key reason the ML setup here is valid.

---

## 3. Feature pipeline built in this notebook

The final feature pipeline includes:

### A. Row-local features
- `moneyness`
- `log_moneyness`
- `maturity_label`
- `maturity_days`
- `tau`
- `option_type`
- center vs wing indicators
- edge vs interior maturity indicators

### B. Same-date anchor / local geometry features
- same-date linear interpolation prediction
- left / right anchor IVs
- left / right anchor distances
- same-maturity visible anchor count
- opposite-option visible IV at the same node
- support counts
- hard-case flag

### C. Historical node features
- `recent_node_pred`
- `full_node_pred`

These act as historical priors for the exact same surface node.

### D. Structured-model features
- `same_date_linear_interp_pred`
- `quadratic_smile_logm_pred`
- `tv_maturity_interp_pred`
- `structured_region_blend_pred`
- `structured_winner_pred`
- differences between structured predictions

### E. Date-level regime proxy features
The problem statement mentions hidden regimes, but this notebook does **not** impose a hard three-state regime model.

Instead, regime information is included as continuous date-level visible-information features:

- `date_atm_iv_proxy`
- `date_skew_proxy`
- `date_term_slope_proxy`
- `date_visible_anchor_ratio`
- `date_visible_iv_dispersion`

This is the right first use of regime information:
- explicit enough to be useful
- but not overcommitted to a hard latent-state model too early

---

## 4. First supervised-learning results

The first modeling comparisons tested:

### Small structured/meta baseline
A compact linear benchmark using mostly:
- structured model outputs
- support features
- opposite-option visibility
- regime proxies

Result:
- it beat the structured winner

Interpretation:
- a lot of useful residual signal is already concentrated in the structured layer itself

---

### Ridge
We tested both:

- direct IV prediction
- residual prediction vs the structured winner

Result:
- both improved materially over the structured winner
- `ridge_direct_full` turned out to be a very strong lightweight reference
- under harsher masking, Ridge direct was especially robust

Interpretation:
- the feature pipeline contains real learnable signal
- even linear models can exploit it well

---

### HistGradientBoosting
We tested:

- direct IV
- residual learning

Result:
- `histgb_residual_full` became the strongest standalone nonlinear model
- nonlinear residual learning worked much better than nonlinear direct prediction

Interpretation:
- the structured winner is a strong prior
- tree models work best when learning the **correction** to that prior rather than the full IV level from scratch

---

### XGBoost
We tested:

- direct IV
- residual learning

Result:
- `xgb_residual_full` was strong and competitive
- but it did not clearly beat `histgb_residual_full`

Interpretation:
- there is real nonlinear residual structure
- but heavier tree models are not automatically better in this dataset

---

## 5. Multi-fold and multi-protocol results

The important step in this notebook was scaling the strongest models across:

- all 4 folds
- both locked protocols:
  - `primary_realistic`
  - `stress_test`

This showed:

- all serious ML models beat the structured winner on both protocols
- `histgb_residual_full` was strongest on `primary_realistic`
- `ridge_direct_full` was strongest on `stress_test`
- `xgb_residual_full` was competitive on both, but not clearly best

This established an important fact:

**The best ML behavior is not uniform across all surface slices or masking regimes.**

That motivated the hybrid routing experiments.

---

## 6. What the slice diagnostics showed

The slice diagnostics were one of the most important parts of the notebook.

### A. Center vs wing
The structured winner remained best in the **center**.

Most ML gains came from the **wings**.

This was the clearest explanation for why naive global ML models were not always optimal.

### B. Hard-case vs non-hard-case
Hard-case counts were small, so those results were informative but not decisive on their own.

Still, they suggested that:
- conservative models helped more in harsher sparse-support settings
- but the overall picture was broader than just hard-case rows

### C. Maturity
The maturity breakdown showed:
- protocol-dependent differences by maturity bucket
- especially under `stress_test`, some maturities favored more conservative models

### D. Structured source buckets
This was extremely useful.

ML improved strongly on:
- weaker/fallback source buckets such as `same_date_linear_interp`
- and several structured-core rows

But it did **not** improve all source buckets equally.
In particular:
- `quadratic_smile_only` rows were often already best handled by the structured winner

This directly motivated the hybrid routing logic.

---

## 7. Hybrid routing experiments

Because the diagnostics showed:
- structured winner best in the center
- ML best in many wing / fallback / difficult slices

we tested rule-based hybrids built from already trained predictions.

### A. `hybrid_center_structured_else_histgb`
Rule:
- use `structured_winner` in the center
- use `histgb_residual_full` elsewhere

Result:
- strongest optimistic hybrid
- best on `primary_realistic`

Interpretation:
- center protection plus nonlinear wing correction is already a strong idea

---

### B. `hybrid_slice_rule_v1`
Rule:
- default to `histgb_residual_full`
- protect:
  - center
  - `quadratic_smile_only`
  with `structured_winner`
- route:
  - `hard_case`
  - `tv_maturity_only`
  to `ridge_direct_full`

Result:
- strongest robust hybrid
- best on `stress_test`

Interpretation:
- conservative overrides help under harsher masking
- but this version turned out to be slightly more complex than necessary

---

## 8. Hybrid ablations and what they taught us

The hybrid ablation pass was decisive.

It showed:

### What mattered
- center protection matters
- `quadratic_smile_only` protection matters
- `tv_maturity_only -> ridge_direct_full` matters

### What did not help enough
- generic `hard_case -> ridge_direct_full` override

Removing the hard-case override improved the balance noticeably.

This produced the final simplified hybrid:

### `hybrid_slice_no_hard_case_override`

Rule:
- default to `histgb_residual_full`
- use `structured_winner` when:
  - `wing_center_bucket == "center"`
  - or `structured_winner_source == "quadratic_smile_only"`
- use `ridge_direct_full` only when:
  - `structured_winner_source == "tv_maturity_only"`

This became the best balanced model in the notebook.

---

## 9. Final Phase 5 leaderboard

The final leaderboard retained the following important models:

### Benchmark
- `structured_winner`

### Best standalone conservative/lightweight model
- `ridge_direct_full`

### Best standalone nonlinear residual model
- `histgb_residual_full`

### Strong residual tree reference
- `xgb_residual_full`

### Best optimistic hybrid
- `hybrid_center_structured_else_histgb`

### Earlier robust hybrid
- `hybrid_slice_rule_v1`

### Current Phase 5 leader
- `hybrid_slice_no_hard_case_override`

---

## 10. Final Phase 5 winner in this notebook

The current Phase 5 leader is:

### `hybrid_slice_no_hard_case_override`

Why this model wins:
- best balanced average across both protocols
- best worst-case protocol score
- clearly improves over the structured winner on both protocols
- simpler and better balanced than the earlier `hybrid_slice_rule_v1`

In summary, this model works because it combines:

- the strong Phase 4 structured prior
- nonlinear residual correction where that helps most
- protection of slices where the structured winner is already strong
- a narrow conservative override only where it is clearly useful

So the final lesson is:

**Selective ML correction works better than replacing the structured model globally.**

---

## 11. Main modeling takeaways from this notebook

### Takeaway 1
The Phase 4 structured winner is strong, but not the ceiling.

### Takeaway 2
Leakage-safe tabular ML adds meaningful value on top of the structured benchmark.

### Takeaway 3
Nonlinear residual learning is more effective than nonlinear direct prediction.

### Takeaway 4
The best model is not a single global ML model.
It is a **hybrid** that preserves the structured model where it is strong and applies ML selectively elsewhere.

### Takeaway 5
Regime information is useful as continuous visible-information features, without yet requiring a hard three-state latent regime model.

---

## 12. What this notebook hands forward

This notebook hands forward:

- a leakage-safe supervised feature pipeline
- reusable train / validation feature builders
- regime/date-level visible-information features
- tested standalone ML references
- tested hybrid routing rules
- a clear current Phase 5 leader:
  - `hybrid_slice_no_hard_case_override`

So the output of this notebook is not only a new score.

It is a clearer modeling picture:

- structured models remain essential
- ML is best used as a controlled correction layer
- routing by slice/source matters
- and the best current Phase 5 model is a simplified hybrid built on top of the structured benchmark
